## Cell 0. API keys

Paste your keys between the quotes below and run this cell before anything
else. Leave a line as `""` to use whatever is already exported in the
environment instead.

**Two things worth knowing before you paste.** This notebook is regenerated by
`build_q1_nb.py`, which rewrites every cell from source -- so a key typed here
is lost on the next rebuild. And a key typed here is saved inside the `.ipynb`
file, where it can reach git or a shared copy. For a key you intend to keep,
put it in `fourarm/env/keys.local.env` instead, which this cell reads
automatically and which is gitignored and never regenerated.

In [1]:
# --- Cell 0. API keys. Run first. -------------------------------------------
import os, pathlib

# PASTE BETWEEN THE QUOTES. Leave "" to fall back to the environment or to
# env/keys.local.env.
KEYS = {
    "OPENAI_API_KEY": "",
    "GEMINI_API_KEY": "",
    "ANTHROPIC_API_KEY": "",
}

# An EMPTY value must never be written into the environment. Assigning ""
# unconditionally would blank a key that is already exported correctly, and
# the failure -- a 401 from a variable that is set but empty -- reads nothing
# like "you left the placeholder alone".
for _name, _value in KEYS.items():
    if _value.strip():
        os.environ[_name] = _value.strip()

# The persistent alternative. Same KEY=value format as env/models.env, one
# per line, # for comments. Read only for names not already set, so anything
# pasted above and anything already exported both win over the file.
_here = pathlib.Path.cwd()
_root = next((c for c in [_here] + list(_here.parents)
              if (c / "out").is_dir() and (c / "experiments").is_dir()), None)
_local = _root / "env" / "keys.local.env" if _root else None
if _local and _local.exists():
    for _line in _local.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip("\'\"")
        if _v and not os.environ.get(_k):
            os.environ[_k] = _v

# Report presence, NEVER the value. Printing a key would write it into the
# notebook's saved output, which is the same leak as pasting it into a cell
# and is easier to do by accident.
#
# The report loops over KEYS, so EVERY key the notebook can use needs a row
# there even when it is only ever supplied by keys.local.env. A name missing
# from KEYS still loads from the file, but silently, and a key that loads
# without being reported is indistinguishable from one that did not load.
for _name in KEYS:
    _set = bool(os.environ.get(_name))
    print("%-18s %s" % (_name, "set" if _set else "NOT SET"))
if _local:
    print("%-18s %s" % ("keys.local.env",
                        "read" if _local.exists() else "absent (optional)"))

OPENAI_API_KEY     set
GEMINI_API_KEY     set
ANTHROPIC_API_KEY  set
keys.local.env     read


# Experiment 2, Q1: Derivation

**When the text omits the capability-relevant quantity, can the model obtain it
from the scene?**

Read at rung **N0** only. The other rungs belong to Q3.

Every cell is independently runnable and idempotent. No cell overwrites a paid
run: the runners resume into their output file and skip trials already
answered. Cells that spend money print the call count and refuse to proceed
until `CONFIRM_SPEND` is set to that exact number.

Run this notebook with the working directory set to `fourarm/`, or anywhere
below it -- cell 1 finds the root itself.

In [2]:
# --- Cell 1. Setup. No model calls. -----------------------------------------
import collections, csv, datetime, hashlib, json, math, os, pathlib, sys

# Find the package root: the directory holding out/ and experiments/.
here = pathlib.Path.cwd()
ROOT = None
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "experiments").is_dir():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("run this from fourarm/ or below: no out/ + experiments/ found")
for p in (str(ROOT), str(ROOT / "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

# RE-IMPORT, never reuse. Python caches modules in sys.modules, so running
# this cell a second time in a live kernel keeps whatever was on disk the
# FIRST time it ran. While the ex2 modules are being edited alongside the
# notebook that is a trap: the kernel holds the old vocabulary, and the
# failure surfaces cells later as a design-check assertion naming a face
# that no longer exists, which reads like a code error and is not one.
#
# Dropping the entries and importing fresh is used rather than
# importlib.reload because these modules import each other, and reload
# leaves a half-updated graph unless the order is exactly right.
for _stale in [m for m in list(sys.modules)
               if m.startswith(("experiments.ex2", "analysis.ex2"))
               or m in ("ycb_objects",)]:
    del sys.modules[_stale]

# WHICH KERNEL THIS IS, checked before the first project import.
#
# The very next line reaches core.decision.state_builder through
# mancheck -> vlm_allocator, and that imports numpy; visibility.py, in cell
# 3, needs PIL and scipy. On a kernel without them the notebook dies forty
# lines deep inside somebody else's module with "No module named 'numpy'",
# which reads as a broken repository rather than as a kernel picked from a
# list of six. Checked here, where the answer is one sentence.
_missing = []
for _m in ("numpy", "PIL", "scipy"):
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    _venv = ROOT.parent / ".venv" / "bin" / "python"
    raise SystemExit(
        "WRONG KERNEL.\n"
        "  This kernel is  %s\n"
        "  and it has no %s.\n"
        "  Use instead     %s\n"
        "  In VS Code: Select Kernel, then Python Environments, then the\n"
        "  interpreter at that path. It is the only one in this tree with\n"
        "  ipykernel AND numpy, PIL and scipy. Several unrelated kernels are\n"
        "  registered on this machine and any of them will get this far and\n"
        "  then fail."
        % (sys.executable, ", ".join(_missing), _venv))

from core.cell import cell_config as C
from core.decision import model_registry as MR
from experiments.ex2 import grade as G
from experiments.ex2 import labels as L
from experiments.ex2 import mancheck as MC
from experiments.ex2 import prompts as P
from experiments.ex2 import run as R
from experiments.ex2 import solo as S
from experiments.ex2 import transforms as T
from experiments.ex2 import visibility as VIS
from analysis.ex2.ex2_stats import newcombe, paired_mean_ci, spans_zero, wilson
# The notebook machinery: loaders, the share definition, the paired
# contrast and the spend gate. In a module rather than in this cell so
# that Q2 and Q3 use the same ones rather than a second copy, and so
# that harness/h_ex2_q_common.py can pin them. What stays in the cells
# is what is a DECISION: the models, the rung, the conditions, the
# usable rule, each cost, and every CONFIRM_SPEND.
from analysis.ex2.ex2_q_common import (Outputs, answered,       # noqa
                                       coupling, fmt, full_flip_count,
                                       is_franka, keep_analysable,
                                       load_run, paired_delta,
                                       paired_diffs, pct,
                                       provenance_row, run_meta,
                                       sha256, share_at, share_counts,
                                       show, spend_gate)

# --- paths ------------------------------------------------------------------
CAPTURES = ROOT / "out" / "ex2_capture_block"
RUNS     = ROOT / "runs"
TABLES   = ROOT / "tables" / "ex2_q1"
FIGURES  = ROOT / "figures" / "ex2_q1"
for d in (RUNS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# --- constants, every one read from a source of truth ------------------------
RUNG        = "N0"                       # Q1 is read here and nowhere else
PREFERENCE  = "franka"
CONDITIONS  = ("congruent", "congruent_face", "dims")
REPEATS     = 3
FACES       = P.RESTING_FACES            # small_face, large_face
LABEL       = "ycb_block"

FRANKA_MAX  = C.ARM_TYPES["franka"]["max_grasp_m"]
UR_MAX      = C.ARM_TYPES["ur10"]["max_grasp_m"]
DIMS        = T.DIMS_M[LABEL]
FACTS       = T.POSE_FACTS_BY_LABEL[LABEL]

# The registry has no hardcoded model list: aliases() reads FOURARM_MODELS.
try:
    ALIASES = MR.aliases()
except Exception as exc:
    ALIASES = []
    print("model registry unavailable (%s); set MODELS by hand below" % exc)
# THREE models since 2026-08-27. claude-sonnet-5 was added because the
# design needs a third model that CLEARS the two-way face probe: with two
# models, a single failure at cell 5b leaves one, and one model cannot show
# that a result is a property of models rather than of this one model.
# It is not here for being the most capable available; see env/models.env.
#
# claude_md RATHER THAN claude. Same model, claude-sonnet-5, at effort
# medium instead of the API default of high. At the default it read the
# two-way face probe at 65 percent against gpt's 95 and gemini's 100, and
# it failed by BIAS rather than blindness: large_face on 75 percent of
# trials, 90 percent right when the block lies flat and 40 percent when it
# stands. Deliberation is how a prior like "blocks lie flat" gains weight,
# so lower effort is the move that fits the failure. Cell 5b is what tests
# it. The default-effort runs stay on disk under the alias "claude".
#
# gpt_hi RATHER THAN gpt. Same model, gpt-5.6-terra, at reasoning_effort
# high instead of low. Claude runs at the Anthropic default effort of high
# and Gemini Flash exposes no effort control at all, so gpt at low made the
# one model with the LEAST test-time compute the yardstick for the other
# two. Effort is still not matched across providers and cannot be -- that
# stays in Limitations -- but the reasoning models are now on the same
# nominal tier.
#
# The low-effort runs are NOT deleted. runs/ex2_q1_cue2way_gpt_r*.jsonl
# record gpt at reasoning_effort low over this same sample and stay on disk
# as the evidence for what effort was worth here: 95 percent at low. A
# separate alias rather than an edit to GPT_PARAMS is what makes those rows
# still readable, which is the reason env/models.env gives for gpt_hi
# existing at all.
#
# Named rather than taken wholesale from ALIASES. FOURARM_MODELS also lists
# qwen and gpt_hi, and a run's model set must be a decision recorded here,
# not whatever the registry happens to carry. The fallback keeps the same
# three so a registry failure cannot silently shrink the design.
_WANT = ("gpt_hi", "gemini", "claude_md")
MODELS = tuple(a for a in ALIASES if a in _WANT) or _WANT
if set(MODELS) != set(_WANT):
    print("WARNING: %s requested, %s available from the registry. Every "
          "table below is per model, so a missing one narrows the design "
          "rather than breaking it -- but say so in the chapter."
          % (list(_WANT), list(MODELS)))

# --- credentials: reported, not assumed -------------------------------------
# Until 2026-08-27 a hand-added launcher cell started JupyterLab in a browser
# and refused to launch when a key was missing. That cell is gone: it spawned
# a NEW server every time it ran, which is how eight of them accumulated, each
# serving its own in-memory copy of this notebook, so an edit on disk could be
# invisible in the tab you were typing in. VS Code runs the kernel directly
# and needs no launcher -- but the key check it performed was worth keeping,
# so it lives here.
#
# This REPORTS rather than raises. Cells 1-5 and every analysis cell make no
# model calls and must stay runnable with no key at all. What it buys is
# learning about a missing key now instead of at cell 6, part-way into a run.
#
# The usual cause is launching VS Code from Finder or the Dock, which does not
# inherit a login shell, so a key exported in .zshrc is absent here while
# present in any terminal. The message below says so, because the symptom
# otherwise looks like a broken registry.
#
# key_var is read from the registry, never hardcoded: models.env lets each
# alias name its own variable, and a hardcoded OPENAI_API_KEY would check the
# wrong one the moment that is used.
MISSING_KEYS = []
for _alias in MODELS:
    try:
        _var = MR.describe(_alias)["key_var"]
    except Exception as _exc:
        MISSING_KEYS.append("%s: %s" % (_alias, _exc))
        continue
    if not os.environ.get(_var):
        MISSING_KEYS.append("%s: %s is not set" % (_alias, _var))

# Output paths travel together in one object, so a notebook cannot end up
# with a root and a tables directory that disagree. Rebound to bare names
# because every call site below reads better as write_csv(...) than as
# OUT.write_csv(...), and because leaving those call sites untouched is
# what made this extraction verifiable against the tables already on disk.
OUT = Outputs(ROOT, TABLES, FIGURES)
rel, write_csv = OUT.rel, OUT.write_csv

print("root        ", ROOT)
print("captures    ", CAPTURES.relative_to(ROOT), "(exists:", CAPTURES.is_dir(), ")")
print("rung        ", RUNG, " preference", PREFERENCE, " repeats", REPEATS)
print("models      ", MODELS, " (registry knows: %s)" % (ALIASES or "nothing"))
print("prompt ver  ", P.EX2_PROMPT_VERSION)
# Printed, not assumed. If a stale kernel ever slips past the re-import
# above, this is the line that shows it, at the top of the run rather than
# in an assertion twenty cells later.
print("faces       ", FACES, " chance %.1f%%" % (100.0 / len(FACES)))
if MISSING_KEYS:
    print("api keys     MISSING -- analysis runs, model calls will not:")
    for _m in MISSING_KEYS:
        print("               ", _m)
    print("             launch VS Code from a shell that exports them:")
    print("               open -a 'Visual Studio Code' <repo>")
else:
    print("api keys     present for %s" % (", ".join(MODELS),))
print()
print("block           %.3f x %.3f x %.3f m"
      % (DIMS["height"], DIMS["width"], DIMS["depth"]))
print("franka opens to  %.3f m   ur opens to %.3f m" % (FRANKA_MAX, UR_MAX))
print("resting faces   %s" % (FACES,))
for f in FACES:
    print("   %-11s needs %.3f m" % (f, FACTS[f]["grasp_m"]))

root         /Users/erinsarlak/Downloads/MastersDissertation/fourarm
captures     out/ex2_capture_block (exists: True )
rung         N0  preference franka  repeats 3
models       ('gpt_hi', 'gemini', 'claude_md')  (registry knows: ['qwen', 'gpt', 'gpt_hi', 'gemini', 'claude', 'claude_md'])
prompt ver   2026-08-27b
faces        ('small_face', 'large_face')  chance 50.0%
api keys     present for gpt_hi, gemini, claude_md

block           0.130 x 0.100 x 0.050 m
franka opens to  0.080 m   ur opens to 0.140 m
resting faces   ('small_face', 'large_face')
   small_face  needs 0.050 m
   large_face  needs 0.100 m


## Cell 2. Design check

No model calls. Derives the opening for each resting face from the authored
cuboid dimensions and asserts it matches what `transforms` declares. The prompt
states the bounding-box convention, so a divergence here would make the prompt
wrong rather than silent.

In [3]:
# --- Cell 2. Design check. No model calls. ----------------------------------
from ycb_objects import YCB as _SPECS      # the authored object dictionary

# STALE-IMPORT GUARD. Cell 1 purges sys.modules before importing, so a
# module edited on disk is picked up whenever cell 1 is re-run. This
# catches the case where cell 1 was NOT re-run -- editing a module and
# jumping straight back to this cell -- and the worse case where the
# NOTEBOOK ITSELF is stale, because JupyterLab holds its own copy in the
# browser and does not re-read the file when it changes underneath. A
# stale cell 1 has no purge, so the modules stay old and the design
# assertion below fails naming a face that no longer exists. That reads
# like a code error and is not one, which is why this checks first and
# says which of the two it is.
#
# The comparison is against the SOURCE ON DISK, not against a constant
# written here, so it stays true across future vocabulary changes.
import re                                   # local: a stale Cell 1 may
                                            # not have imported it
_src = pathlib.Path(P.__file__).read_text()
_on_disk = re.search(r'EX2_PROMPT_VERSION\s*=\s*["\'](.+?)["\']', _src)
if _on_disk and _on_disk.group(1) != P.EX2_PROMPT_VERSION:
    raise SystemExit(
        "STALE IMPORT: this kernel holds prompts.py version %s, but the file "
        "on disk is %s.\n"
        "  Loaded faces: %s\n"
        "  Fix: re-run Cell 1, which drops the cached modules and imports "
        "fresh.\n"
        "  If re-running Cell 1 does not clear it, the NOTEBOOK is stale, not "
        "the kernel:\n"
        "  JupyterLab is running the copy it loaded into the browser. Use "
        "File > Reload\n"
        "  Notebook from Disk, then Restart Kernel and Run All."
        % (P.EX2_PROMPT_VERSION, _on_disk.group(1), list(FACES)))

# Each block prim is spawned already resting on a face, with size PRE-ORIENTED
# to that pose: size is (x, y, z) with z vertical. So the two horizontal
# extents are size[0] and size[1], and the opening is the smaller of them.
# YCB is keyed without the "ycb_" scene prefix.
PRIM_OF_FACE = {L.TRUE_POSE[p]: p for p, lab in L.POSE_ENTRIES.items()
                if lab == LABEL}

design_rows = []
problems = []
for face in FACES:
    prim = PRIM_OF_FACE[face]
    size = _SPECS[prim.replace("ycb_", "")]["size"]
    horiz = sorted(size[:2], reverse=True)          # a = larger, b = smaller
    vertical = size[2]
    opening = min(horiz)
    declared = FACTS[face]["grasp_m"]

    if abs(opening - declared) > 1e-9:
        problems.append("%s: bounding box gives %.3f, transforms declares %.3f"
                        % (face, opening, declared))
    # A real permutation check, all three extents. It compared only the
    # SMALLEST until 2026-08-27, so a prim sized 0.200 x 0.200 x 0.050 --
    # not the block at all -- passed a check whose message said it was
    # verifying a permutation. That matters more with two faces than it
    # did with three: there are fewer cross-checks left, and this cell is
    # what stands between a mis-authored prim and the whole experiment.
    if (sorted(round(v, 6) for v in list(horiz) + [vertical])
            != sorted(round(DIMS[k], 6) for k in ("height", "width", "depth"))):
        problems.append(
            "%s: extents %s are not a permutation of the block %s"
            % (face, sorted(list(horiz) + [vertical]),
               sorted(DIMS[k] for k in ("height", "width", "depth"))))

    franka_ok = declared <= FRANKA_MAX
    ur_ok = declared <= UR_MAX
    design_rows.append([face, "%.3f" % vertical, "%.3f" % horiz[0],
                        "%.3f" % horiz[1], "%.3f" % declared,
                        "%.3f" % FRANKA_MAX, "%.3f" % UR_MAX,
                        franka_ok, ur_ok,
                        "franka, preference satisfied" if franka_ok
                        else "UR, preference overridden"])

# The design only works if the Franka is feasible on one face and not the
# other, and the UR on both. Anything else and Q1 has no contrast.
feasible = [r[0] for r in design_rows if r[7]]
if sorted(feasible) != ["small_face"]:
    problems.append("franka feasible on %s, expected small_face alone"
                    % sorted(feasible))
if not all(r[8] for r in design_rows):
    problems.append("a UR is infeasible somewhere; it must be legal everywhere")

show(["face", "vert", "horiz_a", "horiz_b", "opening", "franka", "ur", "picks"],
     [[r[0], r[1], r[2], r[3], r[4], r[7], r[8], r[9]] for r in design_rows])
print()
if problems:
    raise AssertionError("DESIGN CHECK FAILED:\n  " + "\n  ".join(problems))
print("PASS  every opening is the smaller horizontal extent, and the Franka")
print("      is feasible on small_face and not on large_face.")
print("      A third face, the middle one, was withdrawn on 2026-08-27: it")
print("      was flat like large_face and differed only in geometry, which")
print("      made it the sharper test, but no model read it (GPT 58%,")
print("      Fisher p=0.76 over 81 trials). The cost is that a model")
print("      reading posture and applying a rule can no longer be told")
print("      apart from one deriving the opening from geometry.")

write_csv("tab_ex2_q1_design.csv",
          ["resting_face", "vertical_m", "horiz_a_m", "horiz_b_m",
           "opening_needed_m", "franka_max_m", "ur_max_m", "franka_feasible",
           "ur_feasible", "deriving_model_picks"],
          design_rows)

face        vert   horiz_a  horiz_b  opening  franka  ur    picks                       
----------  -----  -------  -------  -------  ------  ----  ----------------------------
small_face  0.130  0.100    0.050    0.050    True    True  franka, preference satisfied
large_face  0.050  0.130    0.100    0.100    False   True  UR, preference overridden   

PASS  every opening is the smaller horizontal extent, and the Franka
      is feasible on small_face and not on large_face.
      A third face, the middle one, was withdrawn on 2026-08-27: it
      was flat like large_face and differed only in geometry, which
      made it the sharper test, but no model read it (GPT 58%,
      Fisher p=0.76 over 81 trials). The cost is that a model
      reading posture and applying a rule can no longer be told
      apart from one deriving the opening from geometry.
wrote tables/ex2_q1/tab_ex2_q1_design.csv  (2 rows)


PosixPath('/Users/erinsarlak/Downloads/MastersDissertation/fourarm/tables/ex2_q1/tab_ex2_q1_design.csv')

## Cell 3. Capture inventory and legality

No model calls. Loads the captures, checks the grid is complete, re-asserts the
recorded settle heights, and runs the **real validator** over every scene to
establish which positions can carry the contrast at all.

This cell is the reason the sample is 29 positions rather than 30, and it fails
loudly rather than letting the analysis assume otherwise.

**Why the settle heights are re-checked here.** The prompt never says which flat
orientation to expect. The convention sentence -- *an object resting flat lies on
its largest face* -- was deliberately left out, because capture enforces it
instead: `capture_ex2_scene.py` fails a capture that settles at the wrong height
rather than relabelling it with the face it was asked for. That assertion is real
and it does raise, but it post-dates most of the captures on disk, and the trail
check below it compares only the recorded face *word* against the prim -- never
the height that word is supposed to describe. So the single guarantee standing
behind the prompt's silence was being taken on trust at the point where the data
is actually read. Every capture records `ex2.settled[name]`, so checking it costs
nothing. The tolerance is read out of the capture script's source rather than
typed here, so it cannot drift from the value the captures were accepted under.

In [4]:
# --- Cell 3. Capture inventory and legality. No model calls. ----------------
import re                                   # local, as in Cell 2: Cell 1
                                            # does not import it, so this
                                            # cell must not depend on Cell 2
                                            # having been run first
scenes = R.load_scenes(str(CAPTURES))          # normalises the idle UR
raw    = R.load_scenes(str(CAPTURES), present_ur=False)   # as written
trail  = [json.loads(l) for l in open(CAPTURES / "consults.jsonl") if l.strip()]

by_pos = collections.defaultdict(dict)
for s in scenes:
    pos, member = s["seq"].rsplit("_", 1)
    prim = [o["name"] for o in s["state"]["objects"] if LABEL.split("_")[-1] in o["name"]][0]
    by_pos[pos][L.TRUE_POSE[prim]] = s

# DERIVED, never literal. This read "90 captures / 30 positions" until
# 2026-08-27 and raised the moment four positions were added to the capture
# plan. A count typed here goes stale silently; one derived from the
# directory cannot. What actually matters is not the total but that every
# position carries every face, which `missing` below checks.
inv_problems = []
if len(scenes) != len(by_pos) * len(FACES):
    inv_problems.append("expected %d captures (%d positions x %d faces), "
                        "found %d" % (len(by_pos) * len(FACES), len(by_pos),
                                      len(FACES), len(scenes)))
missing = {p: sorted(set(FACES) - set(v)) for p, v in by_pos.items()
           if set(v) != set(FACES)}
if missing:
    inv_problems.append("positions missing a face: %s" % missing)
if len({s["seq"] for s in scenes}) != len(scenes):
    inv_problems.append("duplicate seq ids")

# The face is derived from the PRIM, never from the trail's word, and the
# trail is then checked against it.
#
# Captures written before 2026-08-27 record ex2.resting_face in a superseded
# vocabulary where "upright" meant small_face and "small_face" meant the
# retired middle face. capture_ex2_scene.py now writes the geometric name
# directly, so new captures need no translation; this map reads the old ones
# and is why the check is against the prim rather than the word.
TRAIL_FACE = {"upright": "small_face", "small_face": "edge",
              "large_face": "large_face"}

# READ FROM THE CAPTURE SCRIPT, not typed here. capture_ex2_scene.py
# imports isaaclab at module scope and cannot be imported into this kernel,
# and a literal copied into the notebook would go stale the moment the
# tolerance is retuned -- which it was, from 0.010 to 0.005, on 2026-08-27.
# Same regex-the-source trick cell 2 uses for EX2_PROMPT_VERSION.
_cap_src = (ROOT / "ycb" / "capture_ex2_scene.py").read_text()
_tol = re.search(r"^SETTLE_TOL_M\s*=\s*([0-9.]+)", _cap_src, re.M)
if not _tol:
    raise SystemExit(
        "SETTLE_TOL_M not found in ycb/capture_ex2_scene.py. It is the "
        "tolerance the captures were accepted under and the notebook must "
        "not invent one; if it was renamed, update this cell.")
SETTLE_TOL_M = float(_tol.group(1))
for rec in trail:
    prim = [o["name"] for o in rec["state"]["objects"] if "block" in o["name"]][0]
    want = L.TRUE_POSE.get(prim)
    if want is None:
        continue          # a retired-face capture; load_scenes drops it too
    word = (rec.get("ex2") or {}).get("resting_face")
    # BOTH vocabularies are accepted, and only because each is checked
    # against the prim. A word is fine if it already IS the derived face
    # (written 2026-08-27 or later) or if it translates to it (written
    # before). Anything else is a genuine disagreement. Accepting both is
    # not laxity: the prim is the truth in either case, and the word is
    # never the thing consulted downstream.
    if word != want and TRAIL_FACE.get(word) != want:
        inv_problems.append("%s: trail says %r, prim says %r"
                            % (rec["seq"], word, want))

    # THE SETTLE HEIGHTS, RE-ASSERTED WHERE THE DATA IS READ.
    #
    # The prompt says nothing about which flat orientation to expect. The
    # convention sentence ("an object resting flat lies on its largest
    # face") was deliberately NOT added, on the grounds that capture
    # enforces it instead -- and it does: capture_ex2_scene.py raises on a
    # capture that settles at the wrong height rather than labelling it
    # with the face it was asked for. But that assertion post-dates most
    # captures on disk, and the check above compares only the trail's face
    # WORD against the prim, never the height that word is supposed to
    # describe. So the one guarantee standing behind the prompt's silence
    # was, at this point, taken on trust. It is not expensive to check.
    for name, d in (rec.get("ex2") or {}).get("settled", {}).items():
        z, want_z = d.get("z_above_table"), d.get("expected_rest_z")
        if want_z is None:
            inv_problems.append("%s: %s has no expected_rest_z, so its "
                                "resting face was never verified"
                                % (rec["seq"], name))
        elif abs(z - want_z) > SETTLE_TOL_M:
            inv_problems.append(
                "%s: %s settled at z=%.4f, expected %.4f within %.3f. It is "
                "not on the face this capture claims."
                % (rec["seq"], name, z, want_z, SETTLE_TOL_M))

print("captures %d   positions %d   faces per position %s"
      % (len(scenes), len(by_pos),
         sorted({len(v) for v in by_pos.values()})))
print("idle UR presented: %s"
      % dict(collections.Counter(s["idle_ur"] for s in scenes)))
print("idle UR as captured: %s"
      % dict(collections.Counter(
          tuple(sorted(a["name"] for a in s["state"]["arms"]
                       if a["state"] == "IDLE" and a["name"].startswith("ur")))
          for s in raw)))
print()

# --- the real validator, per scene ------------------------------------------
legal = {}
for pos in sorted(by_pos):
    for face, s in by_pos[pos].items():
        st, meta = T.transform({"state": s["state"],
                                "positions_exact": s["positions_exact"]},
                               "congruent")
        tid = R.flip_task_id(s["state"], meta["flip_prim"])
        legal[(pos, face)] = sorted(R.legal_arms(s, meta["flip_prim"], tid))

def has_franka(arms):
    return any(a.startswith("franka") for a in arms)

# TWO independent preconditions, not one. Legality asks whether the aperture
# contrast EXISTS at a position; visibility asks whether the block can be
# SEEN there. A position can carry the full contrast with the block hidden
# behind the Franka, and until 2026-08-27 nothing noticed: e10 presents 3%
# of the median block area and GPT inverted both its posture trials.
#
# visibility.verdict reads pixels only and never a model reply, so a
# position is never excluded for having scored badly.
#
# Run PER PREFIX, not over the directory at once. Each pose is scored
# against the median of its own pose, and the west and east banks sit at
# different distances from the camera: a w block renders about 10 percent
# larger than an e block in the same pose. One pooled median would raise
# the bar for the far bank and lower it for the near one, which is a
# comparison between banks rather than a test of occlusion.
OCCLUDED, _vis_detail = [], {}
for _pfx in sorted({p[0] for p in by_pos}):
    _c = VIS.measure(str(CAPTURES), prefix=_pfx)
    _ok, _bad, _d = VIS.verdict(_c)
    print("visibility, %s bank:" % _pfx)
    print(VIS.report(_d, _ok, _bad))
    print()
    OCCLUDED += _bad
    _vis_detail.update(_d)

USABLE, excluded = [], {}
for pos in sorted(by_pos):
    sm, lg = (legal[(pos, f)] for f in ("small_face", "large_face"))
    ok = has_franka(sm) and lg and not has_franka(lg)
    if ok:
        USABLE.append(pos)
    else:
        excluded[pos] = {"small_face": sm, "large_face": lg}

show(["position", "small_face", "large_face", "usable"],
     [[pos, ",".join(legal[(pos, "small_face")]) or "NONE",
       ",".join(legal[(pos, "large_face")]) or "NONE",
       "yes" if pos in USABLE else "NO"] for pos in sorted(by_pos)])
print()
USABLE = [p for p in USABLE if p not in OCCLUDED]
print("positions carrying the full contrast and showing the block: %d of %d"
      % (len(USABLE), len(by_pos)))
for pos in sorted(set(OCCLUDED)):
    print("  EXCLUDED %s  the block is occluded here (%.2f of the pose"
          % (pos, _vis_detail[pos]["worst"]))
    print("           median, worst in %s). The contrast may exist, but a"
          % _vis_detail[pos]["worst_pose"])
    print("           perception result from a picture that does not show")
    print("           the object is not a result about the model.")
for pos, v in excluded.items():
    print("  EXCLUDED %s  %s" % (pos, v))
    print("           the franka is never legal here, so there is no arm choice")
    print("           to make and no contrast to measure. Excluded with cause,")
    print("           not dropped silently.")

write_csv("tab_ex2_q1_inventory.csv",
          ["position", "usable", "occluded", "idle_ur", "legal_small_face",
           "legal_large_face"],
          [[pos, pos in USABLE, pos in OCCLUDED,
            by_pos[pos]["small_face"]["idle_ur"],
            ";".join(legal[(pos, "small_face")]),
            ";".join(legal[(pos, "large_face")])] for pos in sorted(by_pos)])

if inv_problems:
    raise AssertionError("INVENTORY FAILED:\n  " + "\n  ".join(inv_problems))
if len(USABLE) < 20:
    raise AssertionError("only %d usable positions; the contrast is not "
                         "estimable and the run should not be paid for"
                         % len(USABLE))

# --- what the PAID cells ask about ------------------------------------------
# Defined here, ONCE, and used by both cell 6 and cell 7. Putting the choice
# in each paid cell would let the two be set differently, so congruent and
# dims would cover different scene sets and the paired contrast in cell 10
# would silently compare two different samples.
#
# TRUE is the standing decision: ask about every captured scene, including
# the excluded positions, and drop them in cell 8 at ANALYSIS time. It costs
# a little more and buys something the write-up needs -- the excluded rows
# are in the data, so the exclusion can be shown to predate any accuracy
# result rather than being read as a position dropped for scoring badly.
#
# Set FALSE to pay only for the usable positions. The analysis is unaffected
# either way: cell 8 restricts to USABLE regardless.
RUN_ALL_POSITIONS = True

CALL_SCENES = ([s for s in scenes
                if s["seq"].rsplit("_", 1)[0] in USABLE]
               if not RUN_ALL_POSITIONS else scenes)

print()
print("PASS  inventory complete, %d positions usable." % len(USABLE))
print("      paid cells will ask about %d scenes (%s)"
      % (len(CALL_SCENES),
         "all captured, excluded positions included on purpose"
         if RUN_ALL_POSITIONS else "usable positions only"))

[ex2] 34 capture(s) skipped: they rest on a face this design no longer uses, and are kept on disk as evidence. e00_S, e01_S, e02_S, e03_S ...
[ex2] 34 capture(s) skipped: they rest on a face this design no longer uses, and are kept on disk as evidence. e00_S, e01_S, e02_S, e03_S ...
captures 68   positions 34   faces per position [2]
idle UR presented: {'ur_w': 34, 'ur_e': 34}
idle UR as captured: {('ur_w',): 68}



visibility, e bank:
pos          L        S        U      worst  verdict
e10        224       51       45       0.03  OCCLUDED (U)
e05        488      893     1064       0.60  ok
e04        719     1380     1258       0.83  ok
e01        650     1273     1325       0.85  ok
e16        655     1311     1302       0.86  ok
e13        661     1327     1356       0.89  ok
e06        685     1438     1390       0.92  ok
e15        720     1483     1549       0.99  ok
e12        736     1497     1509       0.99  ok
e02        717     1499     1518       1.00  ok
e11        727     1495     1530       1.00  ok
e14        774     1551     1567       1.03  ok
e07        758     1562     1600       1.04  ok
e03        798     1564     1644       1.05  ok
e00        811     1650     1625       1.07  ok
e09        828     1709     1703       1.12  ok
e08        840     1741     1738       1.14  ok

cut at 0.50 of the pose median, in every pose.
worst excluded 0.03, best retained 0.60: the cut sits

visibility, w bank:
pos          L        S        U      worst  verdict
w01        646     1266     1326       0.84  ok
w10        663     1326     1374       0.88  ok
w16        669     1347     1363       0.88  ok
w07        679     1360     1369       0.89  ok
w05        687     1432     1424       0.92  ok
w15        719     1491     1503       0.97  ok
w02        720     1486     1517       0.97  ok
w13        720     1494     1541       0.97  ok
w11        768     1505     1507       0.98  ok
w14        741     1564     1563       1.00  ok
w04        758     1622     1595       1.02  ok
w03        787     1559     1587       1.03  ok
w00        802     1650     1620       1.05  ok
w06        800     1602     1621       1.05  ok
w09        803     1623     1627       1.06  ok
w12        824     1705     1717       1.11  ok
w08        855     1774     1757       1.14  ok

cut at 0.50 of the pose median, in every pose.
17 usable, 0 occluded: none
this reads pixels only and never a 

## Cell 4. Prompt inspection

No model calls. Renders the N0 prompt for both conditions and prints them in
full, so what the model reads is on the record beside the numbers it produced.

**dims changes two parts of the prompt, not just the state.** Withholding
`resting_face` and `opening_needed_m` means the field list must stop promising
them and R3 must stop pointing at one of them. R3 is *substituted*, not
deleted: deleting it would test the value of knowing the constraint exists,
which is not the question. A model reading a rule that names a missing field
could reasonably decide the rule is inapplicable, and that would look like a
derivation failure while being a rule-reading failure.

This cell prints both substitutions and asserts neither of them says where the
opening comes from (that is factor C) or calls the absence an error (that
would steer the model toward abstaining, and abstention is one of the
measures).

In [5]:
# --- Cell 4. Prompt inspection. No model calls. -----------------------------
probe_scene = by_pos[USABLE[0]]["large_face"]

rendered = {}
for cond in CONDITIONS:
    msgs, meta = S.render(probe_scene, cond, PREFERENCE, RUNG)
    rendered[cond] = {
        "system": msgs[0]["content"],
        "user": [b for b in msgs[1]["content"] if b.get("type") == "text"][0]["text"],
        "meta": meta}

for cond in CONDITIONS:
    print("=" * 74)
    print("SYSTEM PROMPT  --  %s, rung %s" % (cond.upper(), RUNG))
    print("=" * 74)
    print(rendered[cond]["system"])
    print()

print("=" * 74)
print("DIFF, congruent -> dims  (system prompt)")
print("=" * 74)
import difflib
for line in difflib.unified_diff(rendered["congruent"]["system"].splitlines(),
                                 rendered["dims"]["system"].splitlines(),
                                 lineterm="", n=1):
    if line[:3] not in ("---", "+++", "@@ "):
        print(line)
print()

print("=" * 74)
print("USER MESSAGE  --  dims  (the state the model reads)")
print("=" * 74)
print(rendered["dims"]["user"])
print()

# --- what dims changes in the PROMPT, not just the state --------------------
head = lambda t: t[:t.index("\nYOUR ANSWER")]
r3_of = lambda t: t[t.index("R3  Gripper opening"):t.index("R4  Load")]

print("=" * 74)
print("R3, side by side")
print("=" * 74)
for cond in CONDITIONS:
    print("--- %s" % cond)
    print(r3_of(rendered[cond]["system"]).rstrip())
    print()

print("=" * 74)
print("THE OBJECT FIELD LIST, side by side")
print("=" * 74)
for cond in CONDITIONS:
    print("--- %s" % cond)
    print(P.CONDITIONS[cond]["object_fields"])
    print()

# --- assertions -------------------------------------------------------------
pp = []

# The STATE.
if '"resting_face"' not in rendered["congruent"]["user"]:
    pp.append("congruent must state the resting face")
if '"opening_needed_m"' not in rendered["congruent"]["user"]:
    pp.append("congruent must state the opening")
if '"resting_face"' in rendered["dims"]["user"]:
    pp.append("dims must WITHHOLD the resting face")
if '"opening_needed_m"' in rendered["dims"]["user"]:
    pp.append("dims must WITHHOLD the opening")
if '"size_upright_m"' not in rendered["dims"]["user"]:
    pp.append("dims must keep the object's own dimensions, or nothing is derivable")

# The PROMPT must follow the state. A field list that promises what the state
# does not carry, or a rule pointing at a field that is not there, makes the
# model solve a comprehension puzzle rather than the derivation under test.
dims_head = head(rendered["dims"]["system"])
for f in P.DIMS_WITHHELD:
    if ('"%s"' % f) in dims_head:
        pp.append("the dims field list still promises %s" % f)
    if ('"%s"' % f) not in head(rendered["congruent"]["system"]):
        pp.append("the congruent field list omits %s" % f)

dims_r3 = r3_of(rendered["dims"]["system"])
if "R3  Gripper opening" not in rendered["dims"]["system"]:
    pp.append("R3 must be SUBSTITUTED in dims, never deleted: deleting it "
              "would test the value of knowing the constraint exists")
if '"opening_needed_m"' in dims_r3:
    pp.append("R3 in dims still points at the withheld field")
if "opening_max_m" not in dims_r3:
    pp.append("R3 in dims dropped the capability check itself")
for cond in ("congruent",):
    if '"opening_needed_m"' not in r3_of(rendered[cond]["system"]):
        pp.append("R3 in %s does not name the supplied number" % cond)

# Neither substitution may leak factor C or frame the absence as a fault.
for phrase in ("smaller of", "horizontal extent", "work it out"):
    if phrase in dims_head.lower():
        pp.append("the dims prompt contains %r, which is factor C" % phrase)
for phrase in ("missing", "error", "should have", "incomplete"):
    if phrase in dims_head.lower():
        pp.append("the dims prompt calls the absence %r, which steers the "
                  "model toward abstaining" % phrase)

# The module's own assertions. The glossary and R3 ones are per condition.
for fn in ("assert_base_states_no_relation", "assert_rungs_isolated"):
    try:
        getattr(P, fn)()
        print("PASS  prompts.%s" % fn)
    except Exception as exc:
        pp.append("prompts.%s: %s" % (fn, exc))
for fn in ("assert_glossary_matches_state", "assert_r3_matches_state"):
    for cond in CONDITIONS:
        try:
            getattr(P, fn)(cond)
            print("PASS  prompts.%s(%s)" % (fn, cond))
        except Exception as exc:
            pp.append("prompts.%s(%s): %s" % (fn, cond, exc))

# The registry, the answer schema and the perception probe must speak ONE
# language, or a reported face can never equal the face that was shown.
if L.block_faces() != frozenset(P.RESTING_FACES):
    pp.append("registry faces %s but the schema lists %s"
              % (sorted(L.block_faces()), sorted(P.RESTING_FACES)))

if pp:
    raise AssertionError("PROMPT CHECK FAILED:\n  " + "\n  ".join(pp))
print()
print("PASS  dims withholds BOTH fields from the state.")
print("PASS  the field list stops promising them, and R3 is substituted")
print("      rather than deleted: the capability check survives, the")
print("      pointer to a supplied number does not.")
print("PASS  neither substitution states the relation or calls the absence")
print("      an error.")
print("PASS  N0 carries no factor wording: %r" % P.RUNGS[RUNG]["text"])

SYSTEM PROMPT  --  CONGRUENT, rung N0
You are the task allocator for a four-arm robotic cell. Each time you are
asked, choose ONE queued task and ONE idle arm, or choose to wait.

THE CELL
A 2.8 x 1.6 m table, origin at its centre, x east, y north. Two UR10 arms sit
mid-table on the west and east edges, two Franka arms on the south and north
edges, all facing inward. Zones are a centre disc of radius 0.35 m plus the
quadrants nw, ne, sw, se. Three coloured boxes stand on the table for
sorting, and five exchange points are marked on it.

Every object is picked from directly above. The gripper turns to whichever
horizontal direction suits before it closes. The cell judges an object by the
box that encloses it, so a shape that tapers or curves counts as its full
extent.

WHAT THE STATE TELLS YOU
Each arm states the widest its gripper opens, "opening_max_m", the heaviest
object it can carry, "max_load_kg", and whether it is cleared for delicate
handling, "handles_delicate".

Each object st

## Cell 5. Cue validation

**Makes model calls.** The two-way perception probe, on a sample of positions
across both faces. Chance is one in two.

If the two faces are not separable above chance, the rest of this notebook
should not be run: every later number is a claim about which source the model
believed, and that claim is empty if the image carries no cue.

This probe was three-way until 2026-08-27. The third option, the middle face,
was withdrawn because no model separated it from `large_face` — GPT scored 58%
on that pair, Fisher p = 0.76, over 81 answered trials. The three-way runs are
kept in `runs/ex2_q1_cue_*.jsonl` as the evidence for that decision.

`mancheck.check_id` is `seq|view|model` and carries **no repeat field**, so
repeats go to separate files rather than colliding on resume.

In [6]:
# --- Cell 5. Cue validation. MAKES MODEL CALLS. -----------------------------
# BOTH BANKS. This was USABLE[:10], which is every one of them east,
# because "e" sorts before "w". The banks differ in which UR is idle and in
# how the block sits relative to the camera, and this probe is the gate that
# licenses every paid cell below it, so it should not rest on half the
# workspace. Ten east and ten west: the east ten are the ones already
# collected, so the runners resume and only the west are new.
CUE_POSITIONS = ([p for p in USABLE if p.startswith("e")][:10] +
                 [p for p in USABLE if p.startswith("w")][:10])
CUE_REPEATS = 3

# The filename NAMES THE VOCABULARY, and that is not decoration. mancheck's
# check_id is seq|view|model with no vocabulary in it, so a two-way run
# pointed at a three-way file would find every id already present, make no
# calls, and report the old answers as new. Every id in the retired
# ex2_q1_cue_*.jsonl collides exactly that way. mancheck.assert_same_probe
# refuses it, and this keeps the two sets of runs apart in the first place.
CUE_FILE = "ex2_q1_cue%dway_%%s_r%%d.jsonl" % len(FACES)

cue_seqs = [s["seq"] for pos in CUE_POSITIONS for s in by_pos[pos].values()]
n_calls = len(cue_seqs) * len(MODELS) * CUE_REPEATS
print("COST: %d positions x %d faces x %d models x %d repeats = %d calls"
      % (len(CUE_POSITIONS), len(FACES), len(MODELS), CUE_REPEATS, n_calls))
print("chance is %.1f%%: a %d-way forced choice." % (MC.CHANCE, len(FACES)))
# What this run will ACTUALLY cost, which is not the number above once the
# east half is already on disk. CONFIRM_SPEND is still the full design size,
# because that is what the cell is asking permission for; this line is what
# says how much of it has been bought already.
_have = sum(answered(RUNS / (CUE_FILE % (m, r)))
            for m in MODELS for r in range(1, CUE_REPEATS + 1))
print("already answered: %d across %d files, so this run adds about %d calls"
      % (_have, len(MODELS) * CUE_REPEATS, max(0, n_calls - _have)))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

# No out_path: this cell resumes per model and per repeat inside its own
# loop below, so a single file count would not describe it.
if spend_gate(n_calls, CONFIRM_SPEND,
              factors=(("positions", len(CUE_POSITIONS)),
                       ("faces", len(FACES)), ("models", len(MODELS)),
                       ("repeats", CUE_REPEATS))):
    for model in MODELS:
        for rep in range(1, CUE_REPEATS + 1):
            out = RUNS / (CUE_FILE % (model, rep))
            done = answered(out)
            if done >= len(cue_seqs):
                print("skip %s (%d answered already)" % (out.name, done))
                continue
            print("running %s ..." % out.name)
            MC.run(str(CAPTURES), out_path=str(out), model=model,
                   views=("ex2_cam",), seqs=set(cue_seqs))
    print("\ncue validation complete")

COST: 20 positions x 2 faces x 3 models x 3 repeats = 360 calls
chance is 50.0%: a 2-way forced choice.
already answered: 360 across 9 files, so this run adds about 0 calls
set CONFIRM_SPEND = 360 in this cell to proceed

not confirmed; no calls made.


### Cue validation results

Accuracy per face with Wilson intervals, and the `small_face` / `large_face`
confusion. A verdict, not a table to be interpreted later.

Two-way since 2026-08-27. The probe was three-way and its hard pair was
`edge` against `large_face`; that face was withdrawn because no model read
it. What that costs is printed with the verdict.

In [7]:
# --- Cell 5b. Cue validation results. No model calls. -----------------------
cue_rows = []
for model in MODELS:
    for rep in range(1, CUE_REPEATS + 1):
        f = RUNS / (CUE_FILE % (model, rep))
        if f.exists():
            for line in open(f):
                if line.strip():
                    r = json.loads(line)
                    r["model"], r["repeat"] = model, rep
                    cue_rows.append(r)

# Rows from a probe that offered a different set of answers are not the
# same measurement and must never be pooled with these. The retired
# three-way runs sit in the same directory under ex2_q1_cue_*.jsonl and
# would otherwise be read as if they answered this question: restricted to
# the two surviving faces they score 56 percent, because the model could
# still say "edge", and the gate below would read NOT SEPARABLE and shut
# the notebook for the wrong reason.
_foreign = [r for r in cue_rows
            if (r.get("true_face") or r.get("true_pose")) not in FACES
            or (r.get("answer") is not None and r["answer"] not in FACES)]
if _foreign:
    raise AssertionError(
        "%d of %d cue rows answer a different question (e.g. %r). They are "
        "evidence, not input: read them from their own files. %s holds "
        "only this probe's runs."
        % (len(_foreign), len(cue_rows),
           _foreign[0].get("answer") or _foreign[0].get("true_face"),
           CUE_FILE.replace("%s", "<model>").replace("%d", "<rep>")))

if not cue_rows:
    print("no cue files matching %s yet; run cell 5 first"
          % CUE_FILE.replace("%s", "<model>").replace("%d", "<rep>"))
else:
    CHANCE = 100.0 / len(FACES)
    acc_rows, conf_rows = [], []
    for model in MODELS:
        for face in FACES:
            sub = [r for r in cue_rows if r["model"] == model
                   and (r.get("true_face") or r["true_pose"]) == face
                   and r.get("answer") is not None]
            k = sum(1 for r in sub if r["correct"])
            lo, hi = wilson(k, len(sub))
            acc_rows.append([model, face, len({r["seq"] for r in sub}), k,
                             "%.1f" % (100.0 * k / len(sub)) if sub else "NA",
                             "%.1f" % lo if sub else "NA",
                             "%.1f" % hi if sub else "NA"])
        for tf in FACES:
            for pf in FACES:
                n = sum(1 for r in cue_rows if r["model"] == model
                        and (r.get("true_face") or r["true_pose"]) == tf
                        and r.get("answer") == pf)
                conf_rows.append([model, tf, pf, n])

    show(["model", "face", "images", "correct", "acc%", "lo", "hi"], acc_rows)
    print("\nchance is %.1f%% (%d-way forced choice)" % (CHANCE, len(FACES)))
    write_csv("tab_ex2_q1_cue.csv",
              ["model", "resting_face", "n_images", "correct_n",
               "accuracy_pct", "wilson_lo", "wilson_hi"], acc_rows)
    write_csv("tab_ex2_q1_cue_confusion.csv",
              ["model", "true_face", "predicted_face", "n"], conf_rows)

    # Until 2026-08-27 this block restricted the three-way probe to its
    # hard pair, edge against large_face. With two faces the probe IS that
    # pair, so the restriction is gone and the whole accuracy is the
    # verdict. The Wilson bound, not the point estimate, is what decides:
    # a lower bound above chance is the claim that survives a small n.
    print()
    print("=" * 70)
    print("THE DIAGNOSTIC: small_face against large_face")
    print("=" * 70)
    verdict_ok = True
    for model in MODELS:
        sub = [r for r in cue_rows if r["model"] == model
               and (r.get("true_face") or r["true_pose"]) in FACES
               and r.get("answer") is not None]
        k = sum(1 for r in sub
                if r["answer"] == (r.get("true_face") or r["true_pose"]))
        lo, hi = wilson(k, len(sub))
        ok = lo > CHANCE
        verdict_ok &= ok
        print("  %-8s %d/%d correct = %.1f%% [%.1f, %.1f]   %s"
              % (model, k, len(sub), 100.0 * k / len(sub) if sub else float("nan"),
                 lo, hi, "separable" if ok else "NOT SEPARABLE"))
    print()
    if verdict_ok:
        print("VERDICT  the two faces are separable above chance.")
        print("         Q1's contrast is measurable. Proceed.")
    else:
        print("VERDICT  the two faces are NOT separable.")
        print("         Q1's contrast is DEAD: a null on it would measure")
        print("         the render, not the model. Do not run cells 6 and 7.")
        print("         Report this as an instrument result.")
    print()
    print("  This probe cannot distinguish a model that derives the opening")
    print("  from geometry from one that reads posture and applies a rule:")
    print("  small_face stands and large_face lies, so posture alone scores")
    print("  here. The face that separated those readings was withdrawn on")
    print("  2026-08-27 because no model could see it. State this as a")
    print("  limitation wherever the Q1 verdict is reported.")

model      face        images  correct  acc%   lo    hi   
---------  ----------  ------  -------  -----  ----  -----
gpt_hi     small_face  20      57       95.0   86.3  98.3 
gpt_hi     large_face  20      50       83.3   72.0  90.7 
gemini     small_face  20      60       100.0  94.0  100.0
gemini     large_face  20      60       100.0  94.0  100.0
claude_md  small_face  20      38       63.3   50.7  74.4 
claude_md  large_face  20      57       95.0   86.3  98.3 

chance is 50.0% (2-way forced choice)
wrote tables/ex2_q1/tab_ex2_q1_cue.csv  (6 rows)
wrote tables/ex2_q1/tab_ex2_q1_cue_confusion.csv  (12 rows)

THE DIAGNOSTIC: small_face against large_face
  gpt_hi   107/120 correct = 89.2% [82.3, 93.6]   separable
  gemini   120/120 correct = 100.0% [96.9, 100.0]   separable
  claude_md 95/120 correct = 79.2% [71.1, 85.5]   separable

VERDICT  the two faces are separable above chance.
         Q1's contrast is measurable. Proceed.

  This probe cannot distinguish a model that derive

## Cells 6 and 7. The two conditions at N0

**Make model calls.** Both models, Franka preference, three repeats.
`solo.run` resumes into its output file and skips trials already answered, so
re-running a cell costs nothing and destroys nothing.

The scene count comes from `CALL_SCENES`, set in cell 3. By default that is
**every captured scene**, not just the usable positions: the excluded ones are
asked and then dropped in cell 8 at analysis time, which keeps the exclusion
visible in the data and shows it predates any accuracy result. With 34
captured positions and 32 usable that is 68 scenes rather than 64. Set
`RUN_ALL_POSITIONS = False` in cell 3 to pay only for the usable ones; the
analysis is identical either way.

The count was hardcoded as 90 until 2026-08-27 and had been wrong since the
capture set grew. It is derived now.

In [8]:
# --- Cell 6. Congruent, N0. MAKES MODEL CALLS. ------------------------------
CONGRUENT_OUT = RUNS / "ex2_q1_congruent_N0.jsonl"

# CALL_SCENES, not scenes: cell 3 decides which positions are paid for, so
# this cell and cell 7 cannot drift apart. The breakdown is printed because
# "68 scenes" against "32 usable positions" looks like a bug otherwise, and
# that question is worth answering before the money is spent, not after.
n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
_extra = len(CALL_SCENES) - len(USABLE) * len(FACES)
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      %d usable positions x %d faces = %d scenes%s"
      % (len(USABLE), len(FACES), len(USABLE) * len(FACES),
         ", plus %d at excluded positions, asked so the exclusion is "
         "visible in the data and dropped in cell 8" % _extra
         if _extra else ""))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

# factors= is the guard against a later cell rebinding REPEATS: the gate
# refuses when the counts stop multiplying to the number being confirmed,
# rather than the run quietly coming out a third of the size.
if spend_gate(n_calls, CONFIRM_SPEND, CONGRUENT_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(CONGRUENT_OUT), models=MODELS,
          conditions=("congruent",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(CONGRUENT_OUT))

COST: 68 scenes x 3 models x 3 repeats = 612 calls
      32 usable positions x 2 faces = 64 scenes, plus 4 at excluded positions, asked so the exclusion is visible in the data and dropped in cell 8
already answered: 612 of 612 in ex2_q1_congruent_N0.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cell 6b. Congruent-face at N0

**Makes model calls.** The **true** resting face is stated and
`opening_needed_m` is withheld, so R3 renders in the form that names no field:
*"That opening is not stated for this object."*

This is Q1's question at one remove. `congruent` hands the model the number,
`dims` gives it neither the number nor the face, and this sits between them:
told the face truthfully, can the model get from a face to an opening? A model
that scores here and fails `dims` can do the geometry but cannot read the
orientation off the picture, which is a different failure from one that can do
neither.

It is also the matched ceiling for Q2's `conflict_face`: the two render
byte-identical prompts and differ only in whether the stated face is true.

In [9]:
# --- Cell 6b. Congruent-face, N0. MAKES MODEL CALLS. ------------------------
CONGRUENT_FACE_OUT = RUNS / "ex2_q1_congruent_face_N0.jsonl"
FACE_REPEATS = 3        # bound here, not REPEATS. It was 1 while this cell
                        # was expected to saturate, and gpt_hi and gemini do
                        # saturate. claude_md does not: it sits at chance on
                        # both faces, so its contrast carries real variance
                        # and one repeat cannot bound it. This cell is also
                        # the matched ceiling Q2 reads conflict_face against,
                        # and a ceiling measured at a third of the repeats of
                        # the thing it bounds invites the obvious objection.

n_calls = len(CALL_SCENES) * len(MODELS) * FACE_REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(MODELS), FACE_REPEATS, n_calls))
print("      True face stated, opening withheld. R3 names no field here, so")
print("      the model must derive the opening from the face it is given.")

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, CONGRUENT_FACE_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", FACE_REPEATS))):
    S.run(str(CAPTURES), out_path=str(CONGRUENT_FACE_OUT), models=MODELS,
          conditions=("congruent_face",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=FACE_REPEATS)
    print("answered now:", answered(CONGRUENT_FACE_OUT))

COST: 68 scenes x 3 models x 3 repeat = 612 calls
      True face stated, opening withheld. R3 names no field here, so
      the model must derive the opening from the face it is given.
already answered: 612 of 612 in ex2_q1_congruent_face_N0.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


In [10]:
# --- Cell 7. Dims, N0. MAKES MODEL CALLS. -----------------------------------
DIMS_OUT = RUNS / "ex2_q1_dims_N0.jsonl"

# CALL_SCENES, not scenes: cell 3 decides which positions are paid for, so
# this cell and cell 7 cannot drift apart. The breakdown is printed because
# "68 scenes" against "32 usable positions" looks like a bug otherwise, and
# that question is worth answering before the money is spent, not after.
n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
_extra = len(CALL_SCENES) - len(USABLE) * len(FACES)
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      %d usable positions x %d faces = %d scenes%s"
      % (len(USABLE), len(FACES), len(USABLE) * len(FACES),
         ", plus %d at excluded positions, asked so the exclusion is "
         "visible in the data and dropped in cell 8" % _extra
         if _extra else ""))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, DIMS_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(DIMS_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(DIMS_OUT))

COST: 68 scenes x 3 models x 3 repeats = 612 calls
      32 usable positions x 2 faces = 64 scenes, plus 4 at excluded positions, asked so the exclusion is visible in the data and dropped in cell 8
already answered: 612 of 612 in ex2_q1_dims_N0.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cell 7b. Dims at N0, with no image

**Makes model calls.** The same condition, sample, models and repeat count as
cell 7, with the picture withheld. This is the floor the dims result has to be
read against: whatever a model gets right with no image at all is what the
structured text alone supports.

Under `dims` the two resting faces are **indistinguishable in text**, so the
contrast measured here is zero by construction and what the cell actually
records is how far a model's answer moves when nothing it can see has moved.
The read-out is cell 15, at the end, so that it can reuse cell 8's loader and
cell 9's definition of Franka share rather than keeping a second copy that
could drift from them.

In [11]:
# --- Cell 7b. Dims at N0, NO IMAGE. MAKES MODEL CALLS. ----------------------
NOIMAGE_OUT = RUNS / "ex2_q1_dims_N0_noimage.jsonl"

# THE FLOOR FOR CELL 7. Modality "A" renders the same state and attaches no
# image. Whatever a model gets right here is what the structured text alone
# supports, so cell 10's dims contrast has to be read against it: a contrast
# that survives with no picture was never evidence that the model looked.
#
# THE TWO FACES ARE INDISTINGUISHABLE HERE, BY CONSTRUCTION. dims withholds
# resting_face and opening_needed_m, and size_upright_m is quoted in the
# standing frame whichever way the block actually rests, so at one position
# the only difference between the small_face prompt and the large_face
# prompt is the queued task's id. Checked by rendering both and diffing
# them, not assumed. The expected contrast is therefore ZERO and this cell
# measures how much an answer moves when nothing the model can see moves.
# That is the number cell 10's dims contrast has to be bigger than.
#
# BOTH FACES AND THREE REPEATS ANYWAY, rather than half the calls. Grading,
# legality and the Franka share are keyed on the TRUE pose, which the frozen
# state carries whether or not the text mentions it, so asking under both
# labels is what makes this floor comparable cell for cell with cell 7. Half
# the calls would give a floor computed over a different denominator than
# the number it is a floor for.
#
# ITS OWN FILE. The modality is part of the trial_id, so these rows could
# not collide with cell 7's even inside one file. They are still kept apart,
# because cell 8 reads DIMS_OUT whole: a text-only row landing there would
# be folded into the vision condition and would move every table below
# without appearing anywhere as a decision.
#
# REPEATS IS BOUND LOCALLY, not inherited. Cell 7 rebinds REPEATS for its
# own top-up, so a bare REPEATS here would collect whatever the last cell to
# run happened to leave behind, which is the one way this cell could quietly
# under-sample.
NOIMAGE_REPEATS = 3

n_calls = len(CALL_SCENES) * len(MODELS) * NOIMAGE_REPEATS
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), NOIMAGE_REPEATS, n_calls))
print("      %d usable positions x %d faces = %d scenes, plus the excluded"
      % (len(USABLE), len(FACES), len(USABLE) * len(FACES)))
print("      ones, asked for cell 7's reason and dropped in cell 15.")
print("      No image is attached, so these are the cheapest calls in the")
print("      notebook per trial. solo.cost_table reports what they cost.")

CONFIRM_SPEND = None           # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, NOIMAGE_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", NOIMAGE_REPEATS))):
    S.run(str(CAPTURES), out_path=str(NOIMAGE_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("A",), kind="pair",
          repeats=NOIMAGE_REPEATS)
    print("answered now:", answered(NOIMAGE_OUT))

COST: 68 scenes x 3 models x 3 repeats = 612 calls
      32 usable positions x 2 faces = 64 scenes, plus the excluded
      ones, asked for cell 7's reason and dropped in cell 15.
      No image is attached, so these are the cheapest calls in the
      notebook per trial. solo.cost_table reports what they cost.
already answered: 612 of 612 in ex2_q1_dims_N0_noimage.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cells 7c and 7d. The same two conditions under the `extents` frame

**Make model calls.** The `named` object-field gloss says the extents were
*"measured standing on its smallest face"*. The block is 0.130 x 0.100 x 0.050,
so its smallest face **is** `small_face`: under `dims` that phrase is the only
pose-like statement in the whole prompt, and Q3's frame runs show what it was
worth there. GPT reports the 0.050 opening on 93.1% of `dims` N0 trials under
`named` and 3.4% under `extents`.

`congruent` and `congruent_face` both state `resting_face` outright, so the
phrase is redundant in them and should be inert. **Should is not does**, and
cells 9, 10 and 12 read those two conditions as the reference lines every other
result in the chapter is measured against. These two cells buy the same two
conditions under a gloss that names no orientation, so those reference lines
can be quoted against a prompt that does not contain one of the two answers.

**What the frame changes.** In the system prompt, the gloss: *"its height,
width and depth measured standing on its smallest face, `size_upright_m`"*
becomes *"its three extents largest first, `extents_m`"*. In the user message,
`render_state` renames the field to match. Nothing else moves, and each cell
prints the diff and checks it before it spends: the phrase must go, and the
extents gloss must withhold exactly what the named one withholds. That last
check is not decoration -- until 2026-09-03 `prompts._object_fields` dispatched
on `condition == "dims"` and handed `congruent_face` a gloss announcing an
`opening_needed_m` the condition withholds.

**Matched repeats, read off disk.** The named runs are complete at three
repeats. A frame arm at one repeat would widen every interval by about 1.6x and
leave the clean arm noisier than the confounded one it replaces, so the count
is read from the named twin and asserted rather than taken from `REPEATS`.
That is 612 calls per cell, the same 612 cells 6 and 6b each paid.

**Cell 7c defines the arm; 7d only extends it.** The frame name, the file
naming, the repeat rule and the seeding are stated once, in 7c, so the two
files are one control rather than two that happen to share a suffix. The
read-out is cell 16, at the end, so it can reuse cell 8's loader and cell 10's
contrast rather than keeping a second copy that could drift from them.

In [12]:
# --- Cell 7c. Congruent at N0, EXTENTS frame. MAKES MODEL CALLS. ------------
import difflib

# THE FRAME IS PROMPTS' OWN NAME FOR IT, never a string invented here. If it
# is renamed there this must fail rather than quietly re-buy the DEFAULT
# frame and file it as a control: a file compared with itself reads as a
# clean null, and that is the most expensive way to be wrong in this
# notebook.
FRAME_ALT = "extents"
if FRAME_ALT not in P.DIMS_FRAMES:
    raise AssertionError(
        "%r is not in prompts.DIMS_FRAMES (%s)."
        % (FRAME_ALT, ", ".join(P.DIMS_FRAMES)))

FRAME_CONDS = ("congruent", "congruent_face")

def frame_file(cond):
    """One file per condition, with the frame in the name.

    The frame is in solo's trial_id too, so this is belt and braces -- but a
    frame arm pointed at the named file would find every id present, skip
    the lot, and report itself complete having spent nothing.
    """
    return RUNS / ("ex2_q1_%s_%s_%s.jsonl" % (cond, RUNG, FRAME_ALT))

FRAME_OUT = {c: frame_file(c) for c in FRAME_CONDS}
# The named twin each frame file is read against, named here rather than
# rebuilt in cell 16, so the read-out cannot pair a frame arm with the
# wrong baseline.
FRAME_NAMED = {"congruent": CONGRUENT_OUT,
               "congruent_face": CONGRUENT_FACE_OUT}

def show_frame_edit(cond):
    """Print what the frame changes in the prompt, and check it is that.

    The manipulation is only interpretable if it touches the glossary line
    and nothing else. Two things are asserted rather than described. The
    phrase this control exists to remove must be gone, or the cell is
    buying the confound again under a different file name. And the extents
    gloss must withhold exactly what the named gloss withholds, or the
    frame has changed the CONDITION as well as the wording and the two
    cannot be told apart afterwards.
    """
    a = P.system_prompt(RUNG, cond, dims_frame="named").splitlines()
    b = P.system_prompt(RUNG, cond, dims_frame=FRAME_ALT).splitlines()
    d = list(difflib.unified_diff(a, b, lineterm="", n=0))
    add = [l[1:] for l in d if l.startswith("+") and not l.startswith("+++")]
    rem = [l[1:] for l in d if l.startswith("-") and not l.startswith("---")]
    if not add and not rem:
        raise AssertionError(
            "the %s prompt for %s is identical to the named one. There is "
            "no contrast to buy." % (FRAME_ALT, cond))
    for l in rem:
        if l.strip():
            print("   - " + l)
    for l in add:
        if l.strip():
            print("   + " + l)

    named_gloss = P._object_fields(cond, "named")
    alt_gloss = P._object_fields(cond, FRAME_ALT)
    # WHITESPACE COLLAPSED BEFORE SEARCHING. The gloss is wrapped to the
    # prompt's width, and "standing on its smallest face" falls across a
    # line break in every condition, so a plain substring test finds the
    # phrase absent from the NAMED gloss and reports the control as
    # unnecessary. Checked on the flattened text; printed as it is sent.
    flat = lambda t: " ".join(t.split())
    PHRASE = "smallest face"
    if PHRASE in flat(alt_gloss):
        raise AssertionError(
            "the %s gloss for %s still contains the phrase this control "
            "exists to remove:\n%s" % (FRAME_ALT, cond, alt_gloss))
    if PHRASE not in flat(named_gloss):
        raise AssertionError(
            "the NAMED gloss for %s no longer contains \"%s\", so this "
            "cell is a control against nothing:\n%s"
            % (cond, PHRASE, named_gloss))
    # WITHHELD THE SAME WAY IN BOTH FRAMES. congruent_face and conflict_face
    # withhold the opening and say so; until 2026-09-03 _object_fields
    # dispatched on `condition == "dims"` and handed them the FULL extents
    # gloss, which announces an "opening_needed_m" the state does not carry.
    # That would have changed the condition and the wording at once.
    for field in ("opening_needed_m", "resting_face"):
        if (field in named_gloss) != (field in alt_gloss):
            raise AssertionError(
                "%s names %r in one frame and not the other, so the frame "
                "changes what the condition WITHHOLDS and not only how it "
                "is worded:\n\nnamed:\n%s\n\n%s:\n%s"
                % (cond, field, named_gloss, FRAME_ALT, alt_gloss))
    print()
    print("   PASS  the phrase is gone, and %s withholds the same fields in"
          % cond)
    print("         both frames, so the only difference is the wording.")

def frame_repeats(cond):
    """The named twin's repeat count, read off disk.

    NOT `REPEATS`. congruent is on disk at REPEATS and congruent_face at
    FACE_REPEATS, and the two are free to differ; hard-coding either would
    let the frame arm drift out of match with the arm it is read against,
    and an unmatched control is worth less than no control. A frame arm at
    one repeat would also widen every interval by about 1.6x and leave the
    clean arm noisier than the confounded one it replaces.
    """
    rows, _ = load_run(FRAME_NAMED[cond], cond, MODELS)
    frames = {r.get("dims_frame") or "named" for r in rows}
    if frames - {"named"}:
        raise AssertionError(
            "%s holds %s rows, so it is not the named twin this control is "
            "read against. The two frames must not share a file."
            % (FRAME_NAMED[cond].name, ", ".join(sorted(frames - {"named"}))))
    return len({r.get("repeat") for r in rows if r.get("rung") == RUNG})

# ROWS ALREADY BOUGHT, adopted rather than re-bought.
#
# The 2026-09-03 probe ran exactly these two cells -- same conditions, same
# rung, same frame, one repeat -- through experiments.ex2.launch, one
# process per model, into runs/ex2_q1_frameprobe_N0_extents_<model>.jsonl.
# Those rows are this cell's rows wherever the ALIAS matches: launch ran the
# registry defaults gpt, gemini and claude, and this design runs gpt_hi,
# gemini and claude_md, so gemini's rows are the same measurement and the
# other two are a different model at a different effort. The filter below
# keeps only what MODELS names, which drops them without anyone having to
# remember why.
FRAME_PILOTS = tuple(sorted(RUNS.glob("ex2_q1_frameprobe_%s_%s_*.jsonl"
                                      % (RUNG, FRAME_ALT))))

def seed_frame(cond, sources, repeats):
    """Copy rows this cell would have asked for out of an earlier run.

    Idempotent, and returns what it added and what it refused. THE FILTER
    REBUILDS THE ID THIS CELL WOULD ASK FOR and takes only exact matches,
    for the reason cell 6 of the Q3 notebook does: rebuilding rather than
    pattern-matching means a row from another arm -- another face order,
    another rung, another model alias -- cannot leak in, now or when
    another factor is added later.
    """
    out_path = FRAME_OUT[cond]
    have = set()
    if out_path.exists():
        for line in open(out_path):
            if line.strip():
                have.add(json.loads(line).get("trial_id"))
    added, skipped, take = collections.Counter(), collections.Counter(), []
    for src in sources:
        seen = {}
        for line in open(src):
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except ValueError:
                # A source that is still being written ends in half a line.
                # Skipping it is right and saying so is necessary: a probe
                # still running is a normal thing to seed from.
                skipped["half-written line, source still running"] += 1
                continue
            seen[r.get("trial_id")] = r     # last write wins, as solo does
        for r in seen.values():
            if r.get("condition") != cond:
                continue                    # the other cell's rows
            model = r.get("model")
            if model not in MODELS:
                skipped["model %s, not in this design" % model] += 1
                continue
            if r.get("error"):
                skipped["errored, so never an answer"] += 1
                continue
            rep = r.get("repeat") or 1
            if rep > repeats:
                skipped["beyond this cell's repeats"] += 1
                continue
            want = "%s|%s|%s|%s|%s|V|r%d|%s" % (r.get("seq"), cond, model,
                                                PREFERENCE, RUNG, rep,
                                                FRAME_ALT)
            if r.get("trial_id") != want:
                skipped["a different arm of the same probe"] += 1
                continue
            if r.get("ex2_prompt_version") != P.EX2_PROMPT_VERSION:
                skipped["older prompt version"] += 1
                continue
            if want in have:
                continue
            take.append(r)
            have.add(want)
            added[model] += 1
    # Written only if there is something to write, so a seeding pass that
    # matches nothing does not leave an empty file behind for cell 16 to
    # report as a run that was made and produced no rows.
    if take:
        with open(out_path, "a") as fh:
            for r in take:
                fh.write(json.dumps(r) + "\n")
    return added, skipped

# --- congruent --------------------------------------------------------------
COND = "congruent"
print("=" * 70)
print("THE FRAME EDIT: %s minus named, in %s" % (FRAME_ALT, COND))
print("=" * 70)
show_frame_edit(COND)
print()
print("=" * 70)
print("THE FULL PROMPT SENT, %s %s under %s" % (COND, RUNG, FRAME_ALT))
print("=" * 70)
print(P.system_prompt(RUNG, COND, dims_frame=FRAME_ALT))
print("=" * 70)
print()

FRAME_REPEATS = frame_repeats(COND)
if not FRAME_REPEATS:
    print("The named twin this is read against is not on disk:")
    print("  %s" % rel(FRAME_NAMED[COND]))
    print("Cell 6 buys it. Nothing to control against, nothing bought.")
else:
    _added, _skipped = seed_frame(COND, FRAME_PILOTS, FRAME_REPEATS)
    if _added:
        print("seeded from the 2026-09-03 probe: %s"
              % ", ".join("%s %d" % (m, n) for m, n in sorted(_added.items())))
        for _src in FRAME_PILOTS:
            print("   %s" % rel(_src))
        print("   Not new observations: the same cell, already paid for,")
        print("   resumed rather than re-bought. Cell 14 records where every")
        print("   row in the file came from.")
    if _skipped:
        print("   not seeded: %s"
              % ", ".join("%s %d" % (k, v) for k, v in sorted(_skipped.items())))
    print()

    n_calls = len(CALL_SCENES) * len(MODELS) * FRAME_REPEATS
    print("COST: %d scenes x %d models x %d repeats = %d calls"
          % (len(CALL_SCENES), len(MODELS), FRAME_REPEATS, n_calls))
    print("      condition %s, rung %s, dims frame %s"
          % (COND, RUNG, FRAME_ALT.upper()))
    print("      repeats read from %s, matched to the arm this is read"
          % rel(FRAME_NAMED[COND]))
    print("      against rather than taken from REPEATS.")

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    # factors= is the guard against a later cell rebinding FRAME_REPEATS:
    # the gate refuses when the counts stop multiplying to the number being
    # confirmed, rather than the run quietly coming out a third of the size.
    if spend_gate(n_calls, CONFIRM_SPEND, FRAME_OUT[COND],
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)),
                           ("repeats", FRAME_REPEATS))):
        # dims_frames is the ONE thing that differs from the run that
        # produced the named twin. Scenes, models, preference, rung,
        # modality and kind are read from cells 1 and 3, so the two runs
        # cannot come apart on anything the comparison is not about.
        S.run(str(CAPTURES), out_path=str(FRAME_OUT[COND]), models=MODELS,
              conditions=(COND,), preferences=(PREFERENCE,), rungs=(RUNG,),
              modalities=("V",), kind="pair", repeats=FRAME_REPEATS,
              dims_frames=(FRAME_ALT,))
        print("answered now:", answered(FRAME_OUT[COND]))

THE FRAME EDIT: extents minus named, in congruent
   - is "delicate", its height, width and depth measured standing on its smallest
   - face, "size_upright_m", and the arms that can reach it,
   - "arms_that_can_reach".
   + is "delicate", its three extents largest first, "extents_m", and the arms
   + that can reach it, "arms_that_can_reach".

   PASS  the phrase is gone, and congruent withholds the same fields in
         both frames, so the only difference is the wording.

THE FULL PROMPT SENT, congruent N0 under extents
You are the task allocator for a four-arm robotic cell. Each time you are
asked, choose ONE queued task and ONE idle arm, or choose to wait.

THE CELL
A 2.8 x 1.6 m table, origin at its centre, x east, y north. Two UR10 arms sit
mid-table on the west and east edges, two Franka arms on the south and north
edges, all facing inward. Zones are a centre disc of radius 0.35 m plus the
quadrants nw, ne, sw, se. Three coloured boxes stand on the table for
sorting, and five

## Cell 7d. Congruent-face at N0, under `extents`

**Makes model calls.** The other half of the control. This is the condition the
frame question actually bites on: `congruent` hands the model the opening, so a
gloss that names an orientation has little left to do, while `congruent_face`
states the face and withholds the number, and the model has to get from one to
the other. If the `named` gloss is doing part of that work, this is where it
shows.

It is also the matched ceiling Q2 reads `conflict_face` against, so a frame
effect here does not stop at Q1.

**Nothing about the control is restated here.** The frame name, the file
naming, the repeat rule and the seeding are cell 7c's, used as it left them,
so the two files are one arm rather than two that share a suffix.

In [13]:
# --- Cell 7d. Congruent-face at N0, EXTENTS frame. MAKES MODEL CALLS. -------
# CELL 7c DEFINES THE ARM; this only extends it to the second condition. The
# frame name, the output naming, the repeat rule and the seeding are stated
# once, in 7c, so that this cell cannot buy a differently-defined control
# and file it beside the first. Two arms under one name is worse than one
# arm and a gap.
for _n in ("FRAME_ALT", "FRAME_OUT", "FRAME_NAMED", "frame_repeats",
           "seed_frame", "show_frame_edit", "FRAME_PILOTS"):
    if _n not in globals():
        raise AssertionError(
            "%s is not defined, so cell 7c has not been run in this kernel. "
            "This cell extends 7c's arm rather than restating it." % _n)

COND = "congruent_face"
print("=" * 70)
print("THE FRAME EDIT: %s minus named, in %s" % (FRAME_ALT, COND))
print("=" * 70)
show_frame_edit(COND)
print()
print("=" * 70)
print("THE FULL PROMPT SENT, %s %s under %s" % (COND, RUNG, FRAME_ALT))
print("=" * 70)
print(P.system_prompt(RUNG, COND, dims_frame=FRAME_ALT))
print("=" * 70)
print()

# REPEATS READ FOR THIS CONDITION, not carried over from 7c. congruent is on
# disk at REPEATS and this one at FACE_REPEATS; they are equal today and
# nothing enforces that, so reading again is what keeps each frame arm
# matched to its own twin.
FRAME_REPEATS = frame_repeats(COND)
if not FRAME_REPEATS:
    print("The named twin this is read against is not on disk:")
    print("  %s" % rel(FRAME_NAMED[COND]))
    print("Cell 6b buys it. Nothing to control against, nothing bought.")
else:
    _added, _skipped = seed_frame(COND, FRAME_PILOTS, FRAME_REPEATS)
    if _added:
        print("seeded from the 2026-09-03 probe: %s"
              % ", ".join("%s %d" % (m, n) for m, n in sorted(_added.items())))
        for _src in FRAME_PILOTS:
            print("   %s" % rel(_src))
        print("   Not new observations: the same cell, already paid for,")
        print("   resumed rather than re-bought.")
    if _skipped:
        print("   not seeded: %s"
              % ", ".join("%s %d" % (k, v) for k, v in sorted(_skipped.items())))
    print()

    n_calls = len(CALL_SCENES) * len(MODELS) * FRAME_REPEATS
    print("COST: %d scenes x %d models x %d repeats = %d calls"
          % (len(CALL_SCENES), len(MODELS), FRAME_REPEATS, n_calls))
    print("      condition %s, rung %s, dims frame %s"
          % (COND, RUNG, FRAME_ALT.upper()))
    print("      True face stated, opening withheld, and now no orientation")
    print("      named anywhere in the prompt. This is the cell the frame")
    print("      question bites on.")

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND, FRAME_OUT[COND],
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)),
                           ("repeats", FRAME_REPEATS))):
        S.run(str(CAPTURES), out_path=str(FRAME_OUT[COND]), models=MODELS,
              conditions=(COND,), preferences=(PREFERENCE,), rungs=(RUNG,),
              modalities=("V",), kind="pair", repeats=FRAME_REPEATS,
              dims_frames=(FRAME_ALT,))
        print("answered now:", answered(FRAME_OUT[COND]))

THE FRAME EDIT: extents minus named, in congruent_face
   - "resting_face", its mass, "mass_kg", whether it is "delicate", its height,
   - width and depth measured standing on its smallest face, "size_upright_m", and
   - the arms that can reach it, "arms_that_can_reach". No opening is given.
   + "resting_face", its mass, "mass_kg", whether it is "delicate", its three
   + extents largest first, "extents_m", and the arms that can reach it,
   + "arms_that_can_reach". No opening is given.

   PASS  the phrase is gone, and congruent_face withholds the same fields in
         both frames, so the only difference is the wording.

THE FULL PROMPT SENT, congruent_face N0 under extents
You are the task allocator for a four-arm robotic cell. Each time you are
asked, choose ONE queued task and ONE idle arm, or choose to wait.

THE CELL
A 2.8 x 1.6 m table, origin at its centre, x east, y north. Two UR10 arms sit
mid-table on the west and east edges, two Franka arms on the south and north
edges

## Cell 8. Load and validate replies

No model calls. Nothing is silently dropped: every exclusion is counted and
named, and the rows are kept.

In [14]:
# --- Cell 8. Load and validate. No model calls. -----------------------------
# load_run FILTERS TO MODELS and reports what it skipped; the reasoning is
# in its docstring, in analysis/ex2/ex2_q_common.py, because cell 15 applies
# the same rules to the no-image rows and two copies would drift.
ROWS, SKIPPED_MODELS = [], collections.Counter()
for _path, _cond in ((CONGRUENT_OUT, "congruent"),
                     (CONGRUENT_FACE_OUT, "congruent_face"),
                     (DIMS_OUT, "dims")):
    _r, _s = load_run(_path, _cond, MODELS)
    ROWS += _r
    SKIPPED_MODELS += _s
print("distinct trials loaded: %d  (models %s)"
      % (len(ROWS), ", ".join(MODELS)))
if SKIPPED_MODELS:
    print("not in the design, left in the files and not counted below: %s"
          % ", ".join("%s %d" % (m, n) for m, n in sorted(SKIPPED_MODELS.items())))

flags = collections.Counter()
for r in ROWS:
    if r.get("error"):
        flags["transport error"] += 1
    if r.get("outcome") == "unparseable":
        flags["unparseable reply"] += 1
    if r.get("arm") and r["arm"] not in (r.get("legal_true") or []):
        flags["named an arm the validator rejects"] += 1
    if not r.get("arm"):
        flags["declined (no arm named)"] += 1
    op, arm = r.get("opening_needed_m"), r.get("arm")
    if op is not None and arm:
        cap = C.ARM_TYPES[C.ARMS[arm]["type"]]["max_grasp_m"] if arm in C.ARMS else None
        if cap is not None and op > cap + 1e-9:
            flags["reported an opening its own arm cannot span"] += 1
    if op is None and r.get("arm"):
        flags["named an arm but reported no opening"] += 1

show(["issue", "n"], [[k, v] for k, v in sorted(flags.items())] or [["none", 0]])
print()
print("Nothing above is dropped. The analysis excludes declines from the")
print("Franka-share denominator (cell 9) and reports them in cell 11; every")
print("other flag is carried through so it can be inspected.")

# The same three exclusions cell 15 applies to the no-image rows, from one
# definition. A decline SURVIVES: it is cell 11's numerator, and it is
# excluded from cell 9's denominator there rather than here.
ANALYSED = keep_analysable(ROWS, USABLE)
print()
print("rows after removing errors and unparseables and restricting to the")
print("%d usable positions: %d" % (len(USABLE), len(ANALYSED)))
print("expected: %d positions x %d faces x %d conditions x %d models x %d reps"
      " = %d" % (len(USABLE), len(FACES), len(CONDITIONS), len(MODELS),
                 REPEATS, len(USABLE) * len(FACES) * len(CONDITIONS)
                 * len(MODELS) * REPEATS))

distinct trials loaded: 1836  (models gpt_hi, gemini, claude_md)
issue                               n  
----------------------------------  ---
named an arm the validator rejects  259

Nothing above is dropped. The analysis excludes declines from the
Franka-share denominator (cell 9) and reports them in cell 11; every
other flag is carried through so it can be inspected.

rows after removing errors and unparseables and restricting to the
32 usable positions: 1728
expected: 32 positions x 2 faces x 3 conditions x 3 models x 3 reps = 1728


## Cell 9. Franka share by orientation

Table 2. Declines are excluded from both numerator and denominator; they are
reported separately in cell 11. Wilson bounds are on the proposal denominator.

In [15]:
# --- Cell 9. Franka share by orientation. No model calls. -------------------
share_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            k, n = share_counts(sub)     # n is PROPOSALS, not trials
            lo, hi = wilson(k, n)
            share_rows.append([
                cond, model, face, len({r["position"] for r in sub}), n, k,
                "%.1f" % (100.0 * k / n) if n else "NA",
                "%.1f" % lo if n else "NA", "%.1f" % hi if n else "NA"])

show(["condition", "model", "face", "pos", "proposals", "franka", "share%",
      "lo", "hi"], share_rows)
print()
print("A deriving model reads HIGH on small_face and LOW on large_face.")
print("A model using a posture association reads HIGH, LOW, LOW: it separates")
print("standing from flat but not the two flat faces.")
write_csv("tab_ex2_q1_share.csv",
          ["condition", "model", "resting_face", "n_positions", "n_proposals",
           "franka_n", "franka_share_pct", "wilson_lo", "wilson_hi"],
          share_rows)

condition       model      face        pos  proposals  franka  share%  lo    hi   
--------------  ---------  ----------  ---  ---------  ------  ------  ----  -----
congruent       gpt_hi     small_face  32   96         96      100.0   96.2  100.0
congruent       gpt_hi     large_face  32   96         0       0.0     0.0   3.8  
congruent       gemini     small_face  32   96         96      100.0   96.2  100.0
congruent       gemini     large_face  32   96         0       0.0     0.0   3.8  
congruent       claude_md  small_face  32   96         96      100.0   96.2  100.0
congruent       claude_md  large_face  32   96         0       0.0     0.0   3.8  
congruent_face  gpt_hi     small_face  32   96         87      90.6    83.1  95.0 
congruent_face  gpt_hi     large_face  32   96         1       1.0     0.2   5.7  
congruent_face  gemini     small_face  32   96         96      100.0   96.2  100.0
congruent_face  gemini     large_face  32   96         0       0.0     0.0   3.8  
cong

PosixPath('/Users/erinsarlak/Downloads/MastersDissertation/fourarm/tables/ex2_q1/tab_ex2_q1_share.csv')

## Cell 10. Paired contrasts

Table 3, and the by-position companion so the pairing is inspectable.

**On the interval.** The design document asks for Newcombe. Newcombe is an
interval on the difference of two *independent* proportions; these contrasts
are computed within position and then averaged, so the unit is the position and
the correct interval is a t interval over the 29 paired differences. Both are
emitted: `paired_lo`/`paired_hi` is the one to quote, `newcombe_lo`/
`newcombe_hi` is the unpaired comparison the spec named. They are reported side
by side rather than silently substituted.

In [16]:
# --- Cell 10. Paired contrasts. No model calls. -----------------------------
# ONE contrast. It was two until 2026-08-27, the second being
# edge_minus_large, which is what made the design diagnostic: edge and
# large_face are both flat, so a difference between them could only come
# from geometry. The middle face was withdrawn because no model read it,
# and this is where that loss lands.
CONTRASTS = (("small_minus_large", "small_face", "large_face"),)

bypos_rows, contrast_rows = [], []
ratios = {}
for cond in CONDITIONS:
    for model in MODELS:
        base = [r for r in ANALYSED if r["condition"] == cond
                and r["model"] == model]
        for name, a, b in CONTRASTS:
            # One walk over USABLE feeds both the by-position table and the
            # interval, in one order, so the two cannot disagree about which
            # position is which.
            pairs = paired_diffs(base, USABLE, a, b)
            diffs = [d for _, d in pairs]
            bypos_rows += [[cond, model, pos, name, fmt(d)] for pos, d in pairs]
            mean, plo, phi, npos = paired_mean_ci(diffs)

            # The unpaired comparison the spec asked for, pooled over proposals.
            ka, na = share_counts([r for r in base if r["face"] == a])
            kb, nb = share_counts([r for r in base if r["face"] == b])
            nlo, nhi = newcombe(ka, na, kb, nb)

            # A cell where every position gives the same difference has zero
            # variance, so its t interval collapses to zero width and reads
            # as a precision no sample of 32 supports. The honest quantity is
            # the count of positions that flipped completely, with a Wilson
            # interval on it. Carried in the FILE, not marked by inspection
            # in the chapter, so a reader can trace which cells are saturated
            # and every table that quotes them agrees.
            fk, fn = full_flip_count(diffs)
            flo, fhi = wilson(fk, fn)
            contrast_rows.append([cond, model, name, npos,
                                  "%.1f" % mean if mean == mean else "NA",
                                  "%.1f" % nlo if nlo == nlo else "NA",
                                  "%.1f" % nhi if nhi == nhi else "NA",
                                  "%.1f" % plo if plo == plo else "NA",
                                  "%.1f" % phi if phi == phi else "NA",
                                  spans_zero(plo, phi), "NA",
                                  bool(fn) and fk == fn, fk,
                                  "%.1f" % flo if flo == flo else "NA",
                                  "%.1f" % fhi if fhi == fhi else "NA"])
            ratios[(cond, model, name)] = mean

# condition / congruent, for the one remaining contrast. congruent is the
# baseline and has no ratio to itself; every OTHER condition gets one. It was
# restricted to dims until 2026-08-29, which left congruent_face empty -- and
# congruent_face is the condition sections 2 and 5 both quote a ratio for, so
# the number was being read off arithmetic in the text rather than off a file.
BASELINE = "congruent"
for row in contrast_rows:
    cond, model, name = row[0], row[1], row[2]
    if cond != BASELINE and name == "small_minus_large":
        num = ratios.get((cond, model, name))
        den = ratios.get((BASELINE, model, name))
        # index 10 is the ratio column; 9 is spans_zero. Counted, not guessed:
        # condition, model, contrast, npos, mean, newc_lo, newc_hi,
        # paired_lo, paired_hi, spans_zero, ratio_to_congruent, saturated,
        # flip_n, flip_wilson_lo, flip_wilson_hi.
        row[10] = ("%.2f" % (num / den)) if (den is not None and den == den
                                             and abs(den) > 1e-9
                                             and num is not None and num == num
                                             ) else "NA"

show(["condition", "model", "contrast", "npos", "mean", "newc_lo", "newc_hi",
      "paired_lo", "paired_hi", "spans0", "ratio", "sat", "flips", "flip_lo",
      "flip_hi"], contrast_rows)
write_csv("tab_ex2_q1_contrasts.csv",
          ["condition", "model", "contrast", "n_positions", "mean_diff_pts",
           "newcombe_lo", "newcombe_hi", "paired_lo", "paired_hi",
           "spans_zero", "ratio_to_congruent", "saturated", "flip_n",
           "flip_wilson_lo", "flip_wilson_hi"], contrast_rows)
write_csv("tab_ex2_q1_contrasts_bypos.csv",
          ["condition", "model", "position_id", "contrast", "diff_pts"],
          bypos_rows)

# --- the verdict, against the four patterns in the design document ----------
print()
print("=" * 70)
print("READING, per model, in DIMS")
print("=" * 70)
# THREE patterns, not four. The fourth read "separates standing from flat
# but not the two flat faces: a coarse association, not a derivation", and
# it was the one the design existed to detect. It needed edge_minus_large,
# and the middle face was withdrawn on 2026-08-27 because no model could
# see it. That pattern is now UNTESTABLE, not absent: a model matching it
# reads here as "obtains the opening from the geometry", which is the
# reading it was built to rule out.
#
# Do not let that sit implicitly in the code. It is printed below every
# verdict and belongs in the Limitations section, with runs/ex2_q1_cue_*
# as the evidence that the pattern was real.
for model in MODELS:
    d_sm = ratios.get(("dims", model, "small_minus_large"))
    row = [r for r in contrast_rows if r[0] == "dims" and r[1] == model]
    sm_zero = [r for r in row if r[2] == "small_minus_large"][0][9]
    crow = [r for r in contrast_rows if r[0] == "congruent" and r[1] == model]
    c_zero = [r for r in crow if r[2] == "small_minus_large"][0][9]
    # Branch on the POSITION COUNT, not on spans_zero. spans_zero(nan, nan)
    # is True by design, so a model with no rows at all used to fall through
    # to "cannot obtain it from the scene" -- a reading manufactured from no
    # data, printed in the same words as a real null.
    npos = [r for r in row if r[2] == "small_minus_large"][0][3]

    if not npos:
        v = "NOT RUN. No position carries this contrast for this model."
    elif not sm_zero:
        v = "obtains the opening from the geometry in the scene"
    elif not c_zero:
        v = ("applies the rule when given the opening but cannot obtain it "
             "from the scene")
    else:
        v = ("cannot apply the rule even when given the opening; every later "
             "result for this model is uninterpretable")
    print("  %-8s small-large %s" %
          (model, "NA" if d_sm != d_sm else "%+.1f" % d_sm))
    # A cell where every position gives the same difference has zero
    # variance, so its t interval collapses to zero width and reads as a
    # precision no sample of 32 supports. Say how many positions flipped
    # instead; that is the quantity with an honest interval on it. Read off
    # the ROW rather than recomputed here, so this line and the file cannot
    # disagree about which cells are saturated.
    _row = [r for r in row if r[2] == "small_minus_large"][0]
    if _row[11]:
        print("           saturated: %d of %d positions flipped completely, "
              "Wilson [%s, %s]. Quote that, not the zero-width t interval."
              % (_row[12], npos, _row[13], _row[14]))
    print("           -> %s" % v)
    if not sm_zero:
        print("              CANNOT BE DISTINGUISHED from a model that reads")
        print("              posture and applies a rule. small_face stands")
        print("              and large_face lies, so posture alone produces")
        print("              this result. The contrast that separated the")
        print("              two readings needed a third resting face and")
        print("              was withdrawn: see runs/ex2_q1_cue_*.jsonl.")
        print("              The prompt gives posture a SECOND route to the")
        print("              same answer, independent of the withdrawn face:")
        print("              \"size_upright_m\" names the frame its numbers")
        print("              were taken in, so standing-or-flat fixes the")
        print("              opening and the model never has to work out")
        print("              which two extents are horizontal. Both routes")
        print("              are unavoidable with two faces. Read this")
        print("              verdict as posture-plus-lookup, which geometric")
        print("              derivation would also produce.")

condition       model      contrast           npos  mean   newc_lo  newc_hi  paired_lo  paired_hi  spans0  ratio  sat    flips  flip_lo  flip_hi
--------------  ---------  -----------------  ----  -----  -------  -------  ---------  ---------  ------  -----  -----  -----  -------  -------
congruent       gpt_hi     small_minus_large  32    100.0  94.6     100.0    100.0      100.0      False   NA     True   32     89.3     100.0  
congruent       gemini     small_minus_large  32    100.0  94.6     100.0    100.0      100.0      False   NA     True   32     89.3     100.0  
congruent       claude_md  small_minus_large  32    100.0  94.6     100.0    100.0      100.0      False   NA     True   32     89.3     100.0  
congruent_face  gpt_hi     small_minus_large  32    89.6   80.8     94.0     84.1       95.0       False   0.90   False  22     51.4     82.0   
congruent_face  gemini     small_minus_large  32    100.0  94.6     100.0    100.0      100.0      False   1.00   True   32     89

## Cell 11. Wait rate by orientation

Table 4. The denominator here is **all replies including declines**, unlike the
share table. The two differ on purpose.

Picking a UR on `large_face` is a judgement that the object is too wide.
Declining is a judgement that the model cannot tell. Both matter and they are
not the same.

In [17]:
# --- Cell 11. Wait rate by orientation. No model calls. ---------------------
WAIT_THRESHOLD = 5.0        # percent, below which the table is a sentence

wait_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            k = sum(1 for r in sub if not r.get("arm"))
            lo, hi = wilson(k, len(sub))
            wait_rows.append([cond, model, face, len(sub), k,
                              "%.1f" % (100.0 * k / len(sub)) if sub else "NA",
                              "%.1f" % lo if sub else "NA",
                              "%.1f" % hi if sub else "NA"])

show(["condition", "model", "face", "trials", "declines", "rate%", "lo", "hi"],
     wait_rows)
write_csv("tab_ex2_q1_waits.csv",
          ["condition", "model", "resting_face", "n_trials", "declines_n",
           "decline_rate_pct", "wilson_lo", "wilson_hi"], wait_rows)

rates = [float(r[5]) for r in wait_rows if r[5] != "NA"]
print()
if rates and max(rates) < WAIT_THRESHOLD:
    print("ALL CELLS BELOW %.0f%%. Replace the table with one sentence:" % WAIT_THRESHOLD)
    print('  "Declines were rare throughout, at most %.1f%% in any cell, so'
          % max(rates))
    print('   Franka share reads at face value."')
else:
    print("Waiting is not background. Read the table: a rate rising on")
    print("large_face means the model registers a problem without resolving")
    print("it; a rate rising across dims means it registered the missing")
    print("fields, which is the calibrated response.")

condition       model      face        trials  declines  rate%  lo   hi 
--------------  ---------  ----------  ------  --------  -----  ---  ---
congruent       gpt_hi     small_face  96      0         0.0    0.0  3.8
congruent       gpt_hi     large_face  96      0         0.0    0.0  3.8
congruent       gemini     small_face  96      0         0.0    0.0  3.8
congruent       gemini     large_face  96      0         0.0    0.0  3.8
congruent       claude_md  small_face  96      0         0.0    0.0  3.8
congruent       claude_md  large_face  96      0         0.0    0.0  3.8
congruent_face  gpt_hi     small_face  96      0         0.0    0.0  3.8
congruent_face  gpt_hi     large_face  96      0         0.0    0.0  3.8
congruent_face  gemini     small_face  96      0         0.0    0.0  3.8
congruent_face  gemini     large_face  96      0         0.0    0.0  3.8
congruent_face  claude_md  small_face  96      0         0.0    0.0  3.8
congruent_face  claude_md  large_face  96      0   

## Cell 12. Reported opening

Inline, no table file. `opening_needed_m` is a self-report and never a scored
endpoint on its own, but it localises the failure: a wrong face with a correct
derivation from it is a different failure from a correct face with a wrong
derivation.

In [18]:
# --- Cell 12. Reported opening. No model calls. -----------------------------
TOL = 1e-9
for cond in CONDITIONS:
    print("=" * 70)
    print(cond.upper())
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            true_open = FACTS[face]["grasp_m"]
            stated = [r for r in sub if r.get("opening_needed_m") is not None]
            right = [r for r in stated
                     if abs(r["opening_needed_m"] - true_open) <= 0.006]
            # A wrong number that nevertheless licenses the arm chosen: the
            # model's action follows its own report even though the report is
            # wrong, which is a different failure from acting against it.
            consistent = 0
            for r in stated:
                arm = r.get("arm")
                if not arm or arm not in C.ARMS:
                    continue
                cap = C.ARM_TYPES[C.ARMS[arm]["type"]]["max_grasp_m"]
                if r["opening_needed_m"] <= cap + 1e-9:
                    consistent += 1
            print("  %-8s %-11s stated %2d/%-2d   correct %2d   "
                  "arm consistent with own report %2d"
                  % (model, face, len(stated), len(sub), len(right), consistent))

# COUPLING, BOTH WAYS. This asked only whether the named arm COULD SPAN the
# reported opening until 2026-08-28. A UR opens to 0.140 and every opening
# in this design is 0.050 or 0.100, so every reply naming a UR passed
# automatically, only Franka choices were ever tested, and it read 100
# percent in every cell. A statistic at ceiling whenever the safe arm is
# chosen cannot tell "the arm follows the report" from "this model always
# picks the wide arm", which is exactly the distinction the sentence under
# it claimed to be making.
#
# Agreement is now two-directional, and the two ways of disagreeing mean
# different things, so they are reported apart rather than summed.
print()
print("=" * 70)
print("COUPLING between the reported opening and the arm chosen")
print("=" * 70)
couple_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        sub = [r for r in ANALYSED if r["condition"] == cond
               and r["model"] == model]
        agree, n, over_reach, over_cautious = coupling(sub, FRANKA_MAX)
        lo, hi = wilson(agree, n)
        couple_rows.append([cond, model, n, agree, fmt(pct(agree, n)),
                            fmt(lo), fmt(hi), over_reach, over_cautious])
show(["condition", "model", "coupled", "agree", "agree%", "lo", "hi",
      "said_wide_chose_franka", "said_narrow_chose_ur"], couple_rows)
print()
print("Read the two disagreement columns, not the percentage alone.")
print("  said_wide_chose_franka   the arm cannot close on the opening the")
print("                           model itself reported. Arithmetic, not")
print("                           judgement; grade.self_contradicted counts")
print("                           the same event on the row.")
print("  said_narrow_chose_ur     nothing is violated, but the arm does not")
print("                           follow the report either: an opening a")
print("                           Franka fits, and no Franka named. The old")
print("                           statistic scored every one of these as")
print("                           agreement.")
print()
print("A model whose arm follows its own reported opening is applying the")
print("rule; where it fails, the failure is in obtaining the opening. A model")
print("whose arm contradicts its own report has reasoning and action coming")
print("apart, which is a different finding.")

CONGRUENT
  gpt_hi   small_face  stated 96/96   correct 96   arm consistent with own report 96
  gpt_hi   large_face  stated 96/96   correct 96   arm consistent with own report 96
  gemini   small_face  stated 96/96   correct 96   arm consistent with own report 96
  gemini   large_face  stated 96/96   correct 96   arm consistent with own report 96
  claude_md small_face  stated 96/96   correct 96   arm consistent with own report 96
  claude_md large_face  stated 96/96   correct 96   arm consistent with own report 96
CONGRUENT_FACE
  gpt_hi   small_face  stated 96/96   correct 87   arm consistent with own report 96
  gpt_hi   large_face  stated 96/96   correct 95   arm consistent with own report 96
  gemini   small_face  stated 96/96   correct 96   arm consistent with own report 96
  gemini   large_face  stated 96/96   correct 96   arm consistent with own report 96
  claude_md small_face  stated 96/96   correct 48   arm consistent with own report 96
  claude_md large_face  stated 96/96 

## Cell 13. Figure

Franka share by orientation, every condition and every model, with intervals and
a reference line at the level a model indifferent between arm types would
produce.

Written as plotted values plus a self-contained TikZ picture: matplotlib is not
installed in this project's environments, and a TikZ figure stays editable in
the thesis rather than arriving as a raster. The colours are declared at the
top of the `.tex` so they can be swapped for the thesis `includes.tex` names.

In [19]:
# --- Cell 13. Figure. No model calls. ---------------------------------------
plot_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            m = [r for r in share_rows if r[0] == cond and r[1] == model
                 and r[2] == face]
            if m and m[0][6] != "NA":
                plot_rows.append([cond, model, face, float(m[0][6]),
                                  float(m[0][7]), float(m[0][8])])

fig_csv = FIGURES / "fig_ex2_q1_share.csv"
with open(fig_csv, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["condition", "model", "resting_face", "share_pct",
                "wilson_lo", "wilson_hi"])
    for r in plot_rows:
        w.writerow([r[0], r[1], r[2], "%.1f" % r[3], "%.1f" % r[4], "%.1f" % r[5]])
print("wrote", rel(fig_csv))

# --- TikZ, self-contained, no pgfplots --------------------------------------
PANEL_W, PANEL_H, GAP = 5.2, 4.2, 1.4
BAR_W, GROUP_GAP = 0.42, 0.30
# One colour per model, and EVERY model needs its own. These were two
# entries with a .get(..., "q1blue") default until claude was added on
# 2026-08-27, at which point the default would have drawn claude in
# gemini's blue: a legend naming three models over bars showing two
# colours, which misreads as a duplicated series rather than a missing
# definition. The assertion below is what makes that impossible.
COLOURS = {"gpt_hi": "q1teal", "gpt": "q1teal", "gemini": "q1blue",
           "claude_md": "q1amber", "claude": "q1amber"}
_uncoloured = [m for m in MODELS if m not in COLOURS]
if _uncoloured:
    raise AssertionError(
        "no colour defined for %s. Add one to COLOURS and a matching "
        "\\definecolor below; two models sharing a colour makes the figure "
        "wrong in a way that reads as a result." % _uncoloured)
if len({COLOURS[m] for m in MODELS}) != len(MODELS):
    raise AssertionError("two models share a colour: %s"
                         % {m: COLOURS[m] for m in MODELS})

def y(pct):
    return PANEL_H * pct / 100.0

lines = [
    "% Experiment 2, Q1. Franka share by resting face.",
    "% Generated by notebooks/ex2/ex2_q1_derivation.ipynb -- do not hand-edit.",
    "% Swap the four colour definitions for the thesis includes.tex names.",
    "\\begin{tikzpicture}[x=1cm,y=1cm,font=\\small]",
    "\\definecolor{q1blue}{RGB}{59,110,165}",
    "\\definecolor{q1teal}{RGB}{62,150,146}",
    "\\definecolor{q1amber}{RGB}{198,124,58}",
    "\\definecolor{q1rule}{RGB}{140,140,140}",
]
for pi, cond in enumerate(CONDITIONS):
    x0 = pi * (PANEL_W + GAP)
    lines += [
        "%% --- panel: %s" % cond,
        "\\draw[q1rule] (%.2f,0) -- (%.2f,0);" % (x0, x0 + PANEL_W),
        "\\draw[q1rule] (%.2f,0) -- (%.2f,%.2f);" % (x0, x0, PANEL_H),
        "\\node[anchor=south] at (%.2f,%.2f) {\\textbf{%s}};"
        % (x0 + PANEL_W / 2.0, PANEL_H + 0.15, cond),
    ]
    for gy in (0, 25, 50, 75, 100):
        lines.append("\\draw[q1rule!35] (%.2f,%.2f) -- (%.2f,%.2f);"
                     % (x0, y(gy), x0 + PANEL_W, y(gy)))
        if pi == 0:
            lines.append("\\node[anchor=east,q1rule] at (%.2f,%.2f) {%d};"
                         % (x0 - 0.1, y(gy), gy))
    # a model indifferent between the two arm types names a Franka half the time
    lines.append("\\draw[q1rule,dashed] (%.2f,%.2f) -- (%.2f,%.2f);"
                 % (x0, y(50), x0 + PANEL_W, y(50)))
    for fi, face in enumerate(FACES):
        cx = x0 + PANEL_W * (fi + 0.5) / len(FACES)
        lines.append("\\node[anchor=north,align=center] at (%.2f,-0.12) "
                     "{\\texttt{%s}};" % (cx, face.replace("_", "\\_")))
        for mi, model in enumerate(MODELS):
            m = [r for r in plot_rows if r[0] == cond and r[1] == model
                 and r[2] == face]
            if not m:
                continue
            share, lo, hi = m[0][3], m[0][4], m[0][5]
            bx = cx + (mi - (len(MODELS) - 1) / 2.0) * (BAR_W + 0.06)
            col = COLOURS[model]
            lines.append("\\fill[%s] (%.2f,0) rectangle (%.2f,%.2f);"
                         % (col, bx - BAR_W / 2, bx + BAR_W / 2, y(share)))
            lines.append("\\draw[q1rule,thick] (%.2f,%.2f) -- (%.2f,%.2f);"
                         % (bx, y(lo), bx, y(hi)))
            lines.append("\\draw[q1rule] (%.2f,%.2f) -- (%.2f,%.2f);"
                         % (bx - 0.08, y(lo), bx + 0.08, y(lo)))
            lines.append("\\draw[q1rule] (%.2f,%.2f) -- (%.2f,%.2f);"
                         % (bx - 0.08, y(hi), bx + 0.08, y(hi)))

legx = (len(CONDITIONS) - 1) * (PANEL_W + GAP) + PANEL_W + 0.35
for mi, model in enumerate(MODELS):
    ly = PANEL_H - 0.4 * mi
    lines.append("\\fill[%s] (%.2f,%.2f) rectangle (%.2f,%.2f);"
                 % (COLOURS[model], legx, ly, legx + 0.3, ly + 0.22))
    lines.append("\\node[anchor=west] at (%.2f,%.2f) {%s};"
                 % (legx + 0.38, ly + 0.11, model))
lines.append("\\node[anchor=west,q1rule] at (%.2f,%.2f) "
             "{\\footnotesize indifferent};" % (legx, y(50)))
lines.append("\\node[rotate=90,anchor=south] at (-0.85,%.2f) "
             "{Franka share (\\%%)};" % (PANEL_H / 2.0))
lines.append("\\end{tikzpicture}")

fig_tex = FIGURES / "fig_ex2_q1_share.tex"
fig_tex.write_text("\n".join(lines) + "\n")
print("wrote", rel(fig_tex), "(%d lines)" % len(lines))
print()
print("Compile inside the thesis with \\input{}. It needs only tikz; the four")
print("\\definecolor lines are local so the picture stands alone, and should be")
print("deleted once includes.tex supplies the palette.")

wrote figures/ex2_q1/fig_ex2_q1_share.csv
wrote figures/ex2_q1/fig_ex2_q1_share.tex (130 lines)

Compile inside the thesis with \input{}. It needs only tikz; the four
\definecolor lines are local so the picture stands alone, and should be
deleted once includes.tex supplies the palette.


## Cell 14. Provenance

Every input file with its row count and hash, the prompt version, the model
strings and the date, written beside the tables so any number in the chapter
can be traced back.

In [20]:
# --- Cell 14. Provenance. No model calls. -----------------------------------
prov = []
today = datetime.date.today().isoformat()

for role, path in (("captures", CAPTURES / "consults.jsonl"),
                   ("congruent", CONGRUENT_OUT),
                   ("congruent_face", CONGRUENT_FACE_OUT),
                   ("dims", DIMS_OUT),
                   ("dims_noimage", NOIMAGE_OUT),
                   # The frame arm cells 7c and 7d buy. Listed here rather
                   # than left out because the sha and the row count are how
                   # a reader tells a frame file that was bought whole from
                   # one that was seeded from the 2026-09-03 probe and then
                   # topped up. Both cells define their path before they
                   # spend, so running them with CONFIRM_SPEND unset is
                   # enough to make these rows appear.
                   ("congruent_%s" % FRAME_ALT, FRAME_OUT["congruent"]),
                   ("congruent_face_%s" % FRAME_ALT,
                    FRAME_OUT["congruent_face"])):
    if not pathlib.Path(path).exists():
        prov.append([role, rel(path), 0, "MISSING", "", "", today])
        continue
    n, vers, mods = run_meta(path)
    if role == "captures":
        n = sum(1 for l in open(path) if l.strip())
    prov.append([role, rel(path), n, sha256(path),
                 vers or P.EX2_PROMPT_VERSION, mods, today])

for model in MODELS:
    for rep in range(1, CUE_REPEATS + 1):
        f = RUNS / (CUE_FILE % (model, rep))
        if f.exists():
            prov.append(["cue_%s_r%d" % (model, rep), rel(f),
                         sum(1 for l in open(f) if l.strip()), sha256(f),
                         P.EX2_PROMPT_VERSION, model, today])

show(["role", "rows", "sha256", "prompt_version", "models"],
     [[r[0], r[2], (r[3] or "")[:12], r[4], r[5]] for r in prov])
write_csv("tab_ex2_q1_provenance.csv",
          ["role", "path", "rows", "sha256", "prompt_version", "model_string",
           "run_date"], prov)

print()
print("DESIGN FACTS THAT MUST BE DISCLOSED IN THE CHAPTER")
print("-" * 70)
print("1. The idle UR is chosen per position, as the one that can reach the")
print("   object. Every capture was written with ur_w idle, which is right")
print("   for the west positions and wrong for the east: there the object is")
print("   reached by ur_e, so idle-and-reachable collapsed to franka_n alone")
print("   and large_face had NO legal arm. The arm states are set after the")
print("   frames are rendered and no arm is ever commanded to move, so this")
print("   is a text-layer choice that contradicts nothing in the image.")
print("2. %d positions carry the contrast and show the block. %s excluded"
      % (len(USABLE), ", ".join(sorted(excluded)) or "none"))
print("   for legality: the franka cannot reach, so there is no arm choice.")
print("   %s excluded for occlusion: the block is not visible enough to"
      % (", ".join(sorted(OCCLUDED)) or "none"))
print("   judge, measured from pixels alone and blind to any model reply.")
print("3. Only ex2_cam was captured, so there is no viewpoint control.")
print("4. The design used THREE resting faces until 2026-08-27. The third,")
print("   the middle face, gave the only contrast between two orientations")
print("   that were both flat, and so the only test that separated deriving")
print("   the opening from geometry from reading posture and applying a")
print("   rule. It was withdrawn because no model could see it: GPT scored")
print("   58%% on that pair, Fisher p = 0.76, over 81 answered trials, while")
print("   answering a plain standing-or-flat question 18 times out of 18.")
print("   Evidence: runs/ex2_q1_cue_*.jsonl, retained. CONSEQUENCE: every")
print("   'obtains the opening from the geometry' verdict in this notebook")
print("   is consistent with posture-plus-rule and does not exclude it.")
print("5. Posture reaches the same answer by a SECOND route, which retiring")
print("   the third face did not create and no prompt wording removes.")
print("   'size_upright_m' states the frame its three numbers were taken")
print("   in -- standing on the smallest face -- so with two captured")
print("   faces the object is either in that frame or flat, and a two-way")
print("   posture judgement fixes the opening. The model never has to")
print("   work out which two extents are horizontal.")
print("   NOT PATCHED, deliberately. The extents reach the model as an")
print("   unordered set whatever the field is called, so dropping the")
print("   frame from the gloss would not restore a step; it would add an")
print("   ambiguity, since the numbers could then be read as the extents")
print("   AS PLACED and give min(0.100, 0.050) = 0.050 on large_face --")
print("   the wrong opening on the correct condition, which is")
print("   measurement error rather than a harder task.")
print("   Q1 therefore measures posture-plus-lookup and cannot separate")
print("   it from geometric derivation. State that in Limitations.")
print("6. Contrasts are paired within position; the quoted interval is the")
print("   t interval over positions, not Newcombe. Both are in the CSV.")
print("7. Prompt version %s. Rung %s only." % (P.EX2_PROMPT_VERSION, RUNG))

role                    rows  sha256        prompt_version  models                 
----------------------  ----  ------------  --------------  -----------------------
captures                102   b1579d5d8d1f  2026-08-27b                            
congruent               626   ff8833e493a2  2026-08-27b     claude_md;gemini;gpt_hi
congruent_face          837   853979e16f25  2026-08-27b     claude_md;gemini;gpt_hi
dims                    615   01a58e0f7557  2026-08-27b     claude_md;gemini;gpt_hi
dims_noimage            612   d908dda11931  2026-08-27b     claude_md;gemini;gpt_hi
congruent_extents       612   f3e670145506  2026-08-27b     claude_md;gemini;gpt_hi
congruent_face_extents  618   6f2f1cfb8cb1  2026-08-27b     claude_md;gemini;gpt_hi
cue_gpt_hi_r1           40    d50e8e30ba11  2026-08-27b     gpt_hi                 
cue_gpt_hi_r2           40    d4043653bd8d  2026-08-27b     gpt_hi                 
cue_gpt_hi_r3           40    84e8cc4365b8  2026-08-27b     gpt_hi          

## Cell 15. The no-image floor

No model calls. Reads cell 7b's file and puts it beside the vision result.

Two things are being asked. First, does the dims contrast need the picture:
the floor contrast should be indistinguishable from zero, because the two
prompts differ only in a task id, and a floor that is **not** zero is an
instrument fault rather than a finding. Second, what does a model do when the
opening is genuinely unavailable, since waiting is the defensible answer there
and R3 cannot be satisfied for either arm.

In [21]:
# --- Cell 15. The no-image floor. No model calls. ---------------------------
# Every helper here is the one cells 8, 9 and 10 use, imported from
# analysis/ex2/ex2_q_common.py. A floor computed by a different rule than
# the number it is a floor for is not a floor, and that is now enforced by
# there being one definition rather than a comment promising there are two.
NOIMAGE_ROWS, NOIMAGE_SKIPPED = load_run(NOIMAGE_OUT, "dims_noimage", MODELS)
NOIMAGE = keep_analysable(NOIMAGE_ROWS, USABLE)
print("no-image rows kept: %d   expected %d positions x %d faces x %d models"
      " x %d reps = %d"
      % (len(NOIMAGE), len(USABLE), len(FACES), len(MODELS), NOIMAGE_REPEATS,
         len(USABLE) * len(FACES) * len(MODELS) * NOIMAGE_REPEATS))
if NOIMAGE_SKIPPED:
    print("not in the design, left in the file and not counted: %s"
          % ", ".join("%s %d" % (m, n)
                      for m, n in sorted(NOIMAGE_SKIPPED.items())))

if not NOIMAGE:
    print()
    print("Cell 7b has not been run, so there is no floor to report and the")
    print("dims contrast in cell 10 stands without one. Nothing below runs.")
else:
    # --- share by face, exactly as cell 9 computes it ------------------------
    floor_share = []
    for model in MODELS:
        for face in FACES:
            sub = [r for r in NOIMAGE
                   if r["model"] == model and r["face"] == face]
            k, n = share_counts(sub)
            lo, hi = wilson(k, n)
            floor_share.append([
                model, face, len({r["position"] for r in sub}), len(sub), n,
                len(sub) - n, k,
                "%.1f" % (100.0 * k / n) if n else "NA",
                "%.1f" % lo if n else "NA", "%.1f" % hi if n else "NA"])

    show(["model", "face", "pos", "trials", "proposals", "declines", "franka",
          "share%", "lo", "hi"], floor_share)
    write_csv("tab_ex2_q1_noimage_share.csv",
              ["model", "resting_face", "n_positions", "n_trials",
               "n_proposals", "declines_n", "franka_n", "franka_share_pct",
               "wilson_lo", "wilson_hi"], floor_share)

    # --- the contrast, paired within position as cell 10 pairs it -----------
    print()
    floor_contrast = []
    for model in MODELS:
        base = [r for r in NOIMAGE if r["model"] == model]
        diffs = [d for _, d in paired_diffs(base, USABLE,
                                            "small_face", "large_face")]
        mean, plo, phi, npos = paired_mean_ci(diffs)
        vis = ratios.get(("dims", model, "small_minus_large"))
        floor_contrast.append([
            model, npos,
            "%.1f" % mean if mean == mean else "NA",
            "%.1f" % plo if plo == plo else "NA",
            "%.1f" % phi if phi == phi else "NA",
            spans_zero(plo, phi),
            "%.1f" % vis if (vis is not None and vis == vis) else "NA"])

    show(["model", "npos", "floor", "paired_lo", "paired_hi", "spans0",
          "with_image"], floor_contrast)
    write_csv("tab_ex2_q1_noimage_contrast.csv",
              ["model", "n_positions", "floor_contrast_pts", "paired_lo",
               "paired_hi", "spans_zero", "vision_contrast_pts"],
              floor_contrast)

    # --- what each model's pair of numbers means ----------------------------
    print()
    print("=" * 70)
    print("READING, per model")
    print("=" * 70)
    for row in floor_contrast:
        model, floor_spans, vis = row[0], row[5], row[6]
        vrow = [r for r in contrast_rows
                if r[0] == "dims" and r[1] == model
                and r[2] == "small_minus_large"]
        vis_spans = vrow[0][9] if vrow else True
        print("  %-9s floor %s   with image %s" % (model, row[2], vis))
        if not floor_spans:
            print("           -> INSTRUMENT FAULT, not a result. The two")
            print("              prompts differ only in a task id, so a")
            print("              contrast here cannot come from the design.")
            print("              Do not quote this model's dims contrast")
            print("              until this is explained.")
        elif not vis_spans:
            print("           -> the contrast needs the picture: nothing")
            print("              without it, an effect with it.")
        else:
            print("           -> null either way. This model shows no")
            print("              contrast with the image and none without,")
            print("              so the image is not what it is missing.")

    # --- declining is the defensible answer here ----------------------------
    print()
    print("=" * 70)
    print("DECLINE RATE with no image")
    print("=" * 70)
    print("R3 cannot be satisfied for either arm here: the opening is not")
    print("stated and there is no picture to obtain it from, so waiting is")
    print("the defensible answer and naming an arm is a guess. A model that")
    print("declines is not failing this cell.")
    for model in MODELS:
        sub = [r for r in NOIMAGE if r["model"] == model]
        d = sum(1 for r in sub if not r.get("arm"))
        lo, hi = wilson(d, len(sub))
        print("  %-9s %3d of %3d declined = %5.1f%% [%.1f, %.1f]"
              % (model, d, len(sub), 100.0 * d / len(sub) if sub else float("nan"),
                 lo, hi))

no-image rows kept: 576   expected 32 positions x 2 faces x 3 models x 3 reps = 576
model      face        pos  trials  proposals  declines  franka  share%  lo    hi  
---------  ----------  ---  ------  ---------  --------  ------  ------  ----  ----
gpt_hi     small_face  32   96      96         0         92      95.8    89.8  98.4
gpt_hi     large_face  32   96      96         0         94      97.9    92.7  99.4
gemini     small_face  32   96      96         0         84      87.5    79.4  92.7
gemini     large_face  32   96      96         0         90      93.8    87.0  97.1
claude_md  small_face  32   96      96         0         53      55.2    45.3  64.8
claude_md  large_face  32   96      96         0         56      58.3    48.3  67.7
wrote tables/ex2_q1/tab_ex2_q1_noimage_share.csv  (6 rows)

model      npos  floor  paired_lo  paired_hi  spans0  with_image
---------  ----  -----  ---------  ---------  ------  ----------
gpt_hi     32    -2.1   -7.1       2.9        True    

## Cell 16. The frame control read-out

No model calls. Reads cells 7c and 7d off disk and puts them beside the named
runs cells 9, 10 and 12 report, using the same loader, the same share
definition and the same paired contrast, so the `named` column here reproduces
the numbers already in the Q1 tables rather than being a second,
differently-computed version of them. It checks that it has: a named delta that
disagrees with cell 10's stops the cell.

**Safe to run part-way through a paid cell.** It reads files, not a live
handle, so it survives a kernel restart, and a run still in progress shows a
fractional repeats-worth rather than being silently reported as a null.

**What to read.** Both conditions state the true face, so opening accuracy is
already high under both frames and will not separate the two explanations. What
separates them is whether anything **moves** when the only thing that changed is
a phrase the condition has made redundant.

The anchor column is the diagnostic. Under `dims` the frame moved it by about
90 points: the named gloss names `small_face`, and a model reading that as a
pose claim reports that face's 0.050 opening whatever the picture shows.
Correctness and the anchor come apart only where the true face is `large_face`,
which is exactly half these rows.

**No interval on the frame effect**, on purpose. It is a difference of paired
differences across two runs, and a t interval on it would report a precision
the design does not have. Under `dims` the same quantity moved by roughly 45
points on Gemini at N0 and 30 to 48 on GPT at the treated rungs. Something of
that order is what is being looked for; a handful of points is not resolvable
here and must not be written up as one.

In [22]:
# --- Cell 16. The frame control read-out. No model calls. -------------------
# CELLS 7c AND 7d DEFINE THE ARM. This reads it, and names nothing of its
# own: the frame, the two conditions, the files and the repeat rule all come
# from there, so the read-out cannot end up describing a differently-defined
# control than the one that was bought.
for _n in ("FRAME_ALT", "FRAME_CONDS", "FRAME_OUT", "frame_repeats"):
    if _n not in globals():
        raise AssertionError(
            "%s is not defined, so cell 7c has not been run in this kernel. "
            "Run 7c and 7d -- with CONFIRM_SPEND unset they make no calls "
            "and only define the arm -- then run this." % _n)

# Every helper here is the one cells 8, 9, 10 and 12 use. A control computed
# by a different rule than the number it controls is not a control, and that
# is enforced by there being one definition rather than a comment promising
# there are two.
OPEN_TOL = 0.006                  # the tolerance cell 12 grades openings on
SMALL_OPEN = FACTS["small_face"]["grasp_m"]

# --- what is on disk, reported rather than assumed --------------------------
# A file that is missing or part-written is REPORTED, not silently rendered
# as a blank row. Reading part-way through a paid cell is a normal thing to
# do here, and "the frame arm has not been bought yet" and "it was bought
# and moved nothing" must never look the same.
FRAME_ROWS, frame_status = {}, []
for cond in FRAME_CONDS:
    path = FRAME_OUT[cond]
    per_rep = len(USABLE) * len(FACES) * len(MODELS)
    named_reps = frame_repeats(cond)
    if not pathlib.Path(path).exists():
        FRAME_ROWS[cond] = []
        frame_status.append([cond, path.name, 0, per_rep, "MISSING", "-",
                             named_reps])
        continue
    rows, skipped = load_run(path, cond, MODELS)
    # trial_id carries the frame, so a file can only hold the frame it was
    # written for -- but check rather than assume: an out_path typo would
    # otherwise pool named rows into the frame column and read as a null.
    frames = sorted({r.get("dims_frame") or "named" for r in rows})
    if frames not in ([FRAME_ALT], []):
        raise AssertionError(
            "%s holds dims_frame %s, expected %r. The two frames must not "
            "share a file." % (path.name, frames, FRAME_ALT))
    keep = keep_analysable(rows, USABLE)
    reps = sorted(r for r in {x.get("repeat") for x in keep} if r is not None)
    FRAME_ROWS[cond] = keep
    # A RATIO rather than a complete/partial flag: the arm is bought at the
    # named twin's repeat count, so a flat flag would call a finished arm
    # short whenever that count is more than one.
    frame_status.append([cond, path.name, len(keep), per_rep,
                         "%.2f" % (len(keep) / float(per_rep)),
                         ",".join("r%s" % r for r in reps) or "-", named_reps])
    if skipped:
        print("  %s: %d rows from models outside MODELS, not counted"
              % (path.name, sum(skipped.values())))

show(["condition", "file", "analysable", "per repeat", "repeats worth",
      "repeats present", "named repeats"], frame_status)
print()
print("analysable = no transport error, reply parsed, and the position is one")
print("of the %d that carry the contrast and show the block. A run still in"
      % len(USABLE))
print("progress shows a fractional repeats-worth and fewer repeats than its")
print("named twin; the tables below are still readable, just noisier.")
print()

# --- the per-cell numbers ---------------------------------------------------
def opening_stats(rows):
    """(n stated, % correct for the face SHOWN, % reporting 0.050).

    The third column is the ANCHOR, and it is what the frame moved under
    dims: the named gloss names small_face, so a model reading that phrase
    as a pose claim reports that face's opening whatever the picture shows.
    Correctness and the anchor come apart only where the true face is
    large_face, which is half these rows.
    """
    stated = [r for r in rows if r.get("opening_needed_m") is not None]
    if not stated:
        return 0, None, None
    ok = sum(1 for r in stated
             if abs(r["opening_needed_m"] - FACTS[r["face"]]["grasp_m"])
             <= OPEN_TOL)
    anchored = sum(1 for r in stated
                   if abs(r["opening_needed_m"] - SMALL_OPEN) <= OPEN_TOL)
    return len(stated), pct(ok, len(stated)), pct(anchored, len(stated))

frame_rows, frame_delta = [], {}
for cond in FRAME_CONDS:
    for model in MODELS:
        for frame in ("named", FRAME_ALT):
            sub = [r for r in (ANALYSED if frame == "named"
                               else FRAME_ROWS[cond])
                   if r["condition"] == cond and r["model"] == model]
            n_open, ok, anchor = opening_stats(sub)
            # THE SAME quantity cell 10 reports: Franka share on small_face
            # minus Franka share on large_face, paired within position, the
            # positions then averaged. Not a pooled difference.
            diffs = [d for _, d in paired_diffs(sub, USABLE,
                                                "small_face", "large_face")]
            mean, lo, hi, npos = paired_mean_ci(diffs)
            fk, fn = full_flip_count(diffs)
            frame_delta[(cond, model, frame)] = mean
            frame_rows.append([cond, model, frame, len(sub), n_open, fmt(ok),
                               fmt(anchor),
                               fmt(share_at([r for r in sub
                                             if r["face"] == "small_face"])),
                               fmt(share_at([r for r in sub
                                             if r["face"] == "large_face"])),
                               npos, fmt(mean), fmt(lo), fmt(hi),
                               "%d/%d" % (fk, fn)])

# THE NAMED COLUMN MUST BE CELL 10'S NUMBER. It is computed here from
# ANALYSED with the same helpers rather than read out of `ratios`, so this
# comparison is a real check on both and not a restatement of one of them.
# A disagreement means the two cells are no longer measuring the same thing,
# which would make every frame effect below uninterpretable.
if "ratios" in globals():
    for cond in FRAME_CONDS:
        for model in MODELS:
            a = frame_delta[(cond, model, "named")]
            b = ratios.get((cond, model, "small_minus_large"))
            if a is None or b is None or (a != a and b != b):
                continue
            if abs(a - b) > 1e-9:
                raise AssertionError(
                    "the named delta for %s / %s is %.4f here and %.4f in "
                    "cell 10. The two are meant to be the same quantity "
                    "computed by the same helpers." % (cond, model, a, b))
    print("named column checked against cell 10: identical.")
    print()

show(["condition", "model", "frame", "rows", "stated", "opening ok%",
      "reports .050%", "franka% small", "franka% large", "pos", "delta",
      "lo", "hi", "full flips"], frame_rows)
write_csv("tab_ex2_q1_frame_cells.csv",
          ["condition", "model", "dims_frame", "n_rows", "n_stated",
           "opening_correct_pct", "reports_small_opening_pct",
           "franka_share_small_pct", "franka_share_large_pct", "n_positions",
           "mean_diff_pts", "paired_lo", "paired_hi", "full_flips"],
          frame_rows)
print()
print("delta = franka share on small_face minus franka share on large_face,")
print("        paired within position over %d positions. 100 is the ceiling:"
      % len(USABLE))
print("        the Franka every time the block is on its small face and never")
print("        when it is on its large face. 0 is no contrast at all.")
print()

# --- the one number these two cells exist to produce ------------------------
print("=" * 72)
print("FRAME EFFECT: delta(%s) - delta(named), per condition and model"
      % FRAME_ALT)
print("=" * 72)
move_rows = []
for cond in FRAME_CONDS:
    for model in MODELS:
        dn = frame_delta[(cond, model, "named")]
        de = frame_delta[(cond, model, FRAME_ALT)]
        gap = None if (dn is None or de is None or dn != dn or de != de) \
            else de - dn
        an = [r for r in frame_rows if r[0] == cond and r[1] == model
              and r[2] == "named"][0][6]
        ae = [r for r in frame_rows if r[0] == cond and r[1] == model
              and r[2] == FRAME_ALT][0][6]
        move_rows.append([cond, model, fmt(dn), fmt(de), fmt(gap), an, ae])

show(["condition", "model", "delta named", "delta %s" % FRAME_ALT,
      "difference", ".050% named", ".050%% %s" % FRAME_ALT], move_rows)
write_csv("tab_ex2_q1_frame_effect.csv",
          ["condition", "model", "delta_named_pts", "delta_%s_pts" % FRAME_ALT,
           "difference_pts", "reports_small_opening_named_pct",
           "reports_small_opening_%s_pct" % FRAME_ALT], move_rows)
print()
print("NO INTERVAL ON THE DIFFERENCE COLUMN, on purpose. It is a difference")
print("of paired differences across two runs, and a t interval on it would be")
print("reporting a precision the design does not have. This is a presence")
print("test: under dims the same quantity moved by roughly 45 points on")
print("gemini at N0 and 30 to 48 on gpt at the treated rungs. Something of")
print("that order is the effect being looked for; a handful of points is not")
print("resolvable here and must not be written up as one.")
print()
print("=" * 72)
print("WHAT THE TWO OUTCOMES MEAN, stated before the numbers are read")
print("=" * 72)
print("  NOTHING MOVES     the gloss phrase is inert once a face is stated.")
print("                    The confound is confined to dims, and the")
print("                    reference lines cells 9, 10 and 12 draw from")
print("                    these two conditions stand as they are.")
print()
print("  SOMETHING MOVES   the phrase is read as a pose claim even where a")
print("                    pose is given. The supplied-face conditions are")
print("                    partly text-following, every reference line drawn")
print("                    from them needs qualifying, and so does the")
print("                    matched ceiling Q2 reads conflict_face against.")
print()
for row in move_rows:
    cond, model, gap = row[0], row[1], row[4]
    if gap == "NA":
        print("  %-14s %-9s NOT BOUGHT YET, or no position carries the"
              % (cond, model))
        print("  %-14s %-9s contrast for this model." % ("", ""))
    else:
        print("  %-14s %-9s %s points" % (cond, model, gap))

condition       file                                    analysable  per repeat  repeats worth  repeats present  named repeats
--------------  --------------------------------------  ----------  ----------  -------------  ---------------  -------------
congruent       ex2_q1_congruent_N0_extents.jsonl       576         192         3.00           r1,r2,r3         3            
congruent_face  ex2_q1_congruent_face_N0_extents.jsonl  576         192         3.00           r1,r2,r3         3            

analysable = no transport error, reply parsed, and the position is one
of the 32 that carry the contrast and show the block. A run still in
progress shows a fractional repeats-worth and fewer repeats than its
named twin; the tables below are still readable, just noisier.

named column checked against cell 10: identical.

condition       model      frame    rows  stated  opening ok%  reports .050%  franka% small  franka% large  pos  delta  lo     hi     full flips
--------------  ---------  

---
## Cell 17. Audit: every Q1 table against the thesis

Each table Q1 backs, rebuilt here and checked cell by cell against the number
printed in Chapter 5. A cell that completes silently is a table that still
agrees with the write-up; an edited number breaks the assertion.

This is the Experiment 2 counterpart of section 20 of
`ex1_reproduce_tables.ipynb`. It also emits `tab_ex2_q1_opening.csv`, which
Table 5.8 needed and which cell 12 only printed.

In [23]:
# --- Cell 17. Audit against the thesis. No model calls. ---------------------
# PUBLISHED VALUES, transcribed from Chapter 5 and never computed here. The
# whole point is that they are an independent statement of what the thesis
# says, so deriving them from the same data the cells above use would check
# nothing.

THESIS_56 = {   # Table 5.6, cue validation: correct count and accuracy
    ("gpt_hi", "small_face"): (57, 95.0), ("gpt_hi", "large_face"): (50, 83.3),
    ("gemini", "small_face"): (60, 100.0), ("gemini", "large_face"): (60, 100.0),
    ("claude_md", "small_face"): (38, 63.3), ("claude_md", "large_face"): (57, 95.0),
}
THESIS_56_ALL = {"gpt_hi": 89.2, "gemini": 100.0, "claude_md": 79.2}

THESIS_57 = {   # Table 5.7: share small, share large, contrast, paired CI, flips
    ("congruent", "gpt_hi"):         (100.0, 0.0, 100.0, None, None, 32),
    ("congruent", "gemini"):         (100.0, 0.0, 100.0, None, None, 32),
    ("congruent", "claude_md"):      (100.0, 0.0, 100.0, None, None, 32),
    ("congruent_face", "gpt_hi"):    (90.6, 1.0, 89.6, 84.1, 95.0, 22),
    ("congruent_face", "gemini"):    (100.0, 0.0, 100.0, None, None, 32),
    ("congruent_face", "claude_md"): (47.9, 53.1, -5.2, -21.6, 11.2, 1),
    ("dims", "gpt_hi"):              (94.8, 92.7, 2.1, -6.2, 10.3, 0),
    ("dims", "gemini"):              (97.9, 51.0, 46.9, 34.5, 59.3, 6),
    ("dims", "claude_md"):           (46.9, 57.3, -10.4, -24.3, 3.5, 2),
}

THESIS_58 = {   # Table 5.8, accuracy of the reported opening
    ("congruent", "gpt_hi", "small_face"): 100.0,
    ("congruent", "gpt_hi", "large_face"): 100.0,
    ("congruent", "gemini", "small_face"): 100.0,
    ("congruent", "gemini", "large_face"): 100.0,
    ("congruent", "claude_md", "small_face"): 100.0,
    ("congruent", "claude_md", "large_face"): 100.0,
    ("congruent_face", "gpt_hi", "small_face"): 90.6,
    ("congruent_face", "gpt_hi", "large_face"): 99.0,
    ("congruent_face", "gemini", "small_face"): 100.0,
    ("congruent_face", "gemini", "large_face"): 100.0,
    ("congruent_face", "claude_md", "small_face"): 50.0,
    ("congruent_face", "claude_md", "large_face"): 40.6,
    ("dims", "gpt_hi", "small_face"): 94.8,
    ("dims", "gpt_hi", "large_face"): 7.3,
    ("dims", "gemini", "small_face"): 97.9,
    ("dims", "gemini", "large_face"): 49.0,
    ("dims", "claude_md", "small_face"): 56.2,
    ("dims", "claude_md", "large_face"): 36.5,
}

THESIS_59 = {   # Table 5.9: no-image contrast, paired CI, and the with-image one
    "gpt_hi":    (-2.1, -7.1, 2.9, 2.1),
    "gemini":    (-6.2, -14.3, 1.8, 46.9),
    "claude_md": (-3.1, -15.7, 9.4, -10.4),
}

# A tenth of a point: every value above is printed to one decimal, so this is
# rounding tolerance and nothing more. It is NOT a tolerance on the result.
EPS = 0.06
problems = []


def agrees(label, got, want, eps=EPS):
    if want is None:
        return True
    if got is None or abs(got - want) > eps:
        problems.append("%s: computed %s, thesis prints %s" % (label, got, want))
        return False
    return True


# Read back the CSVs the cells above wrote. Checking the EMITTED table is the
# point: that file is what the chapter's numbers were taken from, so a drift
# between it and the thesis is the drift that matters.
def emitted(name):
    with open(TABLES / name) as fh:
        return list(csv.DictReader(fh))


# --- Table 5.6 -------------------------------------------------------------
cue_csv = {(r["model"], r["resting_face"]): r for r in emitted("tab_ex2_q1_cue.csv")}
for (model, face), (n_right, acc) in sorted(THESIS_56.items()):
    row = cue_csv[(model, face)]
    agrees("5.6 %s %s correct" % (model, face), float(row["correct_n"]),
           float(n_right), 0.5)
    agrees("5.6 %s %s accuracy" % (model, face), float(row["accuracy_pct"]), acc)
for model, acc in sorted(THESIS_56_ALL.items()):
    sub = [r for r in cue_csv.values() if r["model"] == model]
    k = sum(int(r["correct_n"]) for r in sub)
    n = sum(int(r["n_images"]) * 3 for r in sub)      # 20 images x three repeats
    agrees("5.6 %s overall" % model, round(100.0 * k / n, 1) if n else None, acc)

# --- Table 5.7 -------------------------------------------------------------
share_csv = {(r["condition"], r["model"], r["resting_face"]): r
             for r in emitted("tab_ex2_q1_share.csv")}
con_csv = {(r["condition"], r["model"]): r
           for r in emitted("tab_ex2_q1_contrasts.csv")}
for (cond, model), (s_pct, l_pct, d, lo, hi, flips) in sorted(THESIS_57.items()):
    for face, want in (("small_face", s_pct), ("large_face", l_pct)):
        agrees("5.7 %s %s %s share" % (cond, model, face),
               float(share_csv[(cond, model, face)]["franka_share_pct"]), want)
    row = con_csv[(cond, model)]
    agrees("5.7 %s %s contrast" % (cond, model), float(row["mean_diff_pts"]), d)
    agrees("5.7 %s %s flips" % (cond, model), float(row["flip_n"]), float(flips), 0.5)
    if lo is not None:
        agrees("5.7 %s %s paired lo" % (cond, model), float(row["paired_lo"]), lo)
        agrees("5.7 %s %s paired hi" % (cond, model), float(row["paired_hi"]), hi)

# --- Table 5.8, which had no generator -------------------------------------
# Scored against the opening the CAPTURED face implies, at the 6 mm tolerance
# cell 12 uses. The denominator is trials, not replies that stated a number:
# a reply with no opening is a failure to report it, not a missing datum.
opening_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            true_open = FACTS[face]["grasp_m"]
            right = sum(1 for r in sub
                        if r.get("opening_needed_m") is not None
                        and abs(r["opening_needed_m"] - true_open) <= 0.006)
            lo, hi = wilson(right, len(sub))
            pct = round(100.0 * right / len(sub), 1) if sub else None
            opening_rows.append([cond, model, face, len(sub), right, pct,
                                 round(lo, 1), round(hi, 1)])
            agrees("5.8 %s %s %s" % (cond, model, face), pct,
                   THESIS_58.get((cond, model, face)))

print("Table 5.8, accuracy of the reported opening")
show(["condition", "model", "face", "n", "correct", "pct", "lo", "hi"],
     opening_rows)
write_csv("tab_ex2_q1_opening.csv",
          ["condition", "model", "resting_face", "n_trials", "n_correct",
           "correct_pct", "wilson_lo", "wilson_hi"], opening_rows)

# --- Table 5.9 -------------------------------------------------------------
ni_csv = {r["model"]: r for r in emitted("tab_ex2_q1_noimage_contrast.csv")}
for model, (d, lo, hi, vision) in sorted(THESIS_59.items()):
    row = ni_csv[model]
    agrees("5.9 %s no-image contrast" % model, float(row["floor_contrast_pts"]), d)
    agrees("5.9 %s paired lo" % model, float(row["paired_lo"]), lo)
    agrees("5.9 %s paired hi" % model, float(row["paired_hi"]), hi)
    agrees("5.9 %s with-image contrast" % model,
           float(row["vision_contrast_pts"]), vision)

# --- The transcribed constants, against the thesis itself ------------------
# Everything above compares the emitted CSVs to the constants at the top of
# this cell. That verifies the pipeline still reproduces them and nothing
# more: the constants are typed from the chapter, so a mis-transcription would
# pass, and a change to the chapter would go unnoticed. This checks them
# against the thesis LaTeX, in the order it prints them.
import thesis_check as _TC

# One vendored file per notebook. A shared one would be clobbered by
# whichever notebook ran last, since each knows only its own tables.
_CK = _TC.Checker(os.path.join(ROOT, "notebooks", "ex2",
                               "thesis_expected_q1.json"))
_covered = []

# 5.6. The wrong-count columns are 60 minus the right count: 20 images at
# three repeats.
_seq = []
for _m in ("gpt_hi", "gemini", "claude_md"):
    _ns, _as = THESIS_56[(_m, "small_face")]
    _nl, _al = THESIS_56[(_m, "large_face")]
    _seq += [_ns, 60 - _ns, _as, 60 - _nl, _nl, _al, THESIS_56_ALL[_m]]
_covered.append(("5.6", "tab:ex2:cue") + _CK.check_subsequence("tab:ex2:cue", _seq))

# 5.7. Share on each face, the contrast, then its interval where one is
# printed; saturated cells print a dagger instead.
_seq = []
for _c in ("congruent", "congruent_face", "dims"):
    for _m in ("gpt_hi", "gemini", "claude_md"):
        _s, _l, _d, _lo, _hi, _f = THESIS_57[(_c, _m)]
        _seq += [_s, _l, _d] + ([_lo, _hi] if _lo is not None else []) + [_f]
_covered.append(("5.7", "tab:ex2:q1:results")
                + _CK.check_subsequence("tab:ex2:q1:results", _seq))

# 5.8. Accuracy on each face. The thesis collapses congruent to a single
# "all three" row, because all three models sit at 100 there, and every cell
# carries a Wilson interval whose bounds this sequence skips over.
_seq = [THESIS_58[("congruent", "gpt_hi", "small_face")],
        THESIS_58[("congruent", "gpt_hi", "large_face")]]
for _c in ("congruent_face", "dims"):
    for _m in ("gpt_hi", "gemini", "claude_md"):
        _seq += [THESIS_58[(_c, _m, "small_face")],
                 THESIS_58[(_c, _m, "large_face")]]
_covered.append(("5.8", "tab:ex2:q1:reported")
                + _CK.check_subsequence("tab:ex2:q1:reported", _seq))

# 5.9. Both no-image shares, the contrast, its interval, the with-image one.
_seq = []
for _m in ("gpt_hi", "gemini", "claude_md"):
    _d, _lo, _hi, _v = THESIS_59[_m]
    _row = [r for r in noimage_share_rows if r[0] == _m] if "noimage_share_rows" in dir() else []
    _seq += [_d, _lo, _hi, _v]
_covered.append(("5.9", "tab:ex2:q1:noimage")
                + _CK.check_subsequence("tab:ex2:q1:noimage", _seq))

print()
for _num, _lab, _got, _tot in _covered:
    print("  %-5s %-22s %d of %d thesis cells pinned%s"
          % (_num, _lab, _got, _tot,
             "" if _got == _tot else "   <-- the rest are not checked here"))
print(_CK.refresh())

# --- Verdict ---------------------------------------------------------------
CHECKED = {"5.6": len(THESIS_56) * 2 + len(THESIS_56_ALL),
           "5.7": len(THESIS_57) * 5, "5.8": len(THESIS_58),
           "5.9": len(THESIS_59) * 4}
print("Experiment 2, Q1: tables checked against the thesis")
show(["Table", "Reports", "Checks"],
     [["5.6", "cue validation, both faces and overall", CHECKED["5.6"]],
      ["5.7", "Franka share by face, paired contrast, complete flips", CHECKED["5.7"]],
      ["5.8", "accuracy of the reported opening", CHECKED["5.8"]],
      ["5.9", "the no-image floor", CHECKED["5.9"]]])

print()
for p in problems:
    print("  MISMATCH  %s" % p)
print("%d checks against Chapter 5, %d disagreed"
      % (sum(CHECKED.values()), len(problems)))
assert not problems, "Q1 no longer reproduces the thesis: %s" % problems[:5]
print("every Q1 table still matches what Chapter 5 prints")

Table 5.8, accuracy of the reported opening


condition       model      face        n   correct  pct    lo    hi   
--------------  ---------  ----------  --  -------  -----  ----  -----
congruent       gpt_hi     small_face  96  96       100.0  96.2  100.0
congruent       gpt_hi     large_face  96  96       100.0  96.2  100.0
congruent       gemini     small_face  96  96       100.0  96.2  100.0
congruent       gemini     large_face  96  96       100.0  96.2  100.0
congruent       claude_md  small_face  96  96       100.0  96.2  100.0
congruent       claude_md  large_face  96  96       100.0  96.2  100.0
congruent_face  gpt_hi     small_face  96  87       90.6   83.1  95.0 
congruent_face  gpt_hi     large_face  96  95       99.0   94.3  99.8 
congruent_face  gemini     small_face  96  96       100.0  96.2  100.0
congruent_face  gemini     large_face  96  96       100.0  96.2  100.0
congruent_face  claude_md  small_face  96  48       50.0   40.2  59.8 
congruent_face  claude_md  large_face  96  39       40.6   31.3  50.6 
dims 